In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — CONFIGURATION
# Paper: "Brain Tumor Classification in MRI: Insights From
#         LIME and Grad-CAM Explainable AI Techniques"
# IEEE Document 11330252
# =============================================================

import os

# ─────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────
SEED = 42

# ─────────────────────────────────────────────
# DATASET PATHS
# ─────────────────────────────────────────────
# Kaggle paths (primary)
KAGGLE_DATASET_ROOT = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
KAGGLE_TRAIN_DIR    = os.path.join(KAGGLE_DATASET_ROOT, "Training")
KAGGLE_TEST_DIR     = os.path.join(KAGGLE_DATASET_ROOT, "Testing")

# Local paths (fallback for development)
#LOCAL_DATASET_ROOT  = os.path.join(os.path.dirname(__file__), "..", "dataset")
#LOCAL_TRAIN_DIR     = os.path.join(LOCAL_DATASET_ROOT, "Training")
#LOCAL_TEST_DIR      = os.path.join(LOCAL_DATASET_ROOT, "Testing")

# Output directory for saved models, plots, and results
OUTPUT_DIR = "/kaggle/working/paper_replication_outputs"

# ─────────────────────────────────────────────
# DATA / PREPROCESSING HYPERPARAMETERS (from paper Table 3 & Section III-A)
# ─────────────────────────────────────────────
IMG_SIZE        = (224, 224)   # bilinear interpolation resize to 224×224
IMG_CHANNELS    = 3
INPUT_SHAPE     = IMG_SIZE + (IMG_CHANNELS,)  # (224, 224, 3)
NUM_CLASSES     = 4
CLASS_NAMES     = ['glioma', 'meningioma', 'notumor', 'pituitary']
VALIDATION_SPLIT = 0.20        # 80:20 train:val split

# ─────────────────────────────────────────────
# AUGMENTATION PARAMETERS (paper Section III-A, Table 3)
# Applied to TRAINING set ONLY
# ─────────────────────────────────────────────
ROTATION_RANGE       = 15      # ±15°
BRIGHTNESS_RANGE     = [0.9, 1.1]
ZOOM_RANGE           = 0.10    # ±10%
WIDTH_SHIFT_RANGE    = 0.10    # ±10%
HEIGHT_SHIFT_RANGE   = 0.10    # ±10%
FILL_MODE            = 'nearest'
RESCALE              = 1.0 / 255.0  # Min-Max Normalisation → [0, 1]

# ─────────────────────────────────────────────
# TRAINING HYPERPARAMETERS (paper Table 5)
# ─────────────────────────────────────────────
BATCH_SIZE           = 8
EPOCHS               = 150
LEARNING_RATE        = 1e-4     # Adam initial lr
DROPOUT_RATE         = 0.30

# Early Stopping (paper Table 5)
EARLY_STOP_MONITOR   = 'val_loss'
EARLY_STOP_PATIENCE  = 20
EARLY_STOP_DELTA     = 0.0
RESTORE_BEST_WEIGHTS = True

# ─────────────────────────────────────────────
# FINE-TUNING HYPERPARAMETERS (beyond paper — prevents overfitting)
# ─────────────────────────────────────────────
PHASE1_EPOCHS        = 20      # frozen-base warm-up phase
FINE_TUNE_LR         = 1e-5    # lower lr for fine-tuning phase
L2_REG               = 1e-4    # L2 regularisation on Dense layers
LABEL_SMOOTHING      = 0.10    # label smoothing

# ReduceLROnPlateau (fine-tuning aid)
REDUCE_LR_FACTOR     = 0.5
REDUCE_LR_PATIENCE   = 5
REDUCE_LR_MIN_LR     = 1e-7

# ──────────────────────��──────────────────────
# MODEL IDENTIFIERS
# ─────────────────────────────────────────────
MODEL_NAMES = ['vgg16', 'resnet50', 'efficientnetb0']

# ─────────────────────────────────────────────
# PAPER ACCURACY TARGETS (for reference / reporting)
# ─────────────────────────────────────────────
PAPER_TARGETS = {
    'vgg16': {
        'overall': 98.78,
        'per_class': {'glioma': 98.00, 'meningioma': 98.04,
                      'notumor': 100.00, 'pituitary': 98.67},
        'early_stop_epoch': 48,
    },
    'resnet50': {
        'overall': 98.25,
        'per_class': {'glioma': 95.00, 'meningioma': 98.69,
                      'notumor': 99.75, 'pituitary': 99.00},
        'early_stop_epoch': 61,
    },
    'efficientnetb0': {
        'overall': 95.58,
        'per_class': {'glioma': 95.00, 'meningioma': 89.22,
                      'notumor': 98.02, 'pituitary': 99.33},
        'early_stop_epoch': 143,
    },
}

In [9]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — DATA PREPROCESSING
# Paper: "Brain Tumor Classification in MRI: Insights From
#         LIME and Grad-CAM Explainable AI Techniques"
# =============================================================

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ✅ FIXED: Replaced "from .config import ..." with inline values
# ─────────────────────────────────────────────
# CONFIG VALUES (hardcoded for Kaggle notebook)
# ─────────────────────────────────────────────
SEED               = 42
IMG_SIZE           = (240, 240)
BATCH_SIZE         = 32
CLASS_NAMES        = ['glioma', 'meningioma', 'notumor', 'pituitary']
VALIDATION_SPLIT   = 0.2
ROTATION_RANGE     = 20
BRIGHTNESS_RANGE   = [0.8, 1.2]
ZOOM_RANGE         = 0.2
WIDTH_SHIFT_RANGE  = 0.2
HEIGHT_SHIFT_RANGE = 0.2
FILL_MODE          = 'nearest'
RESCALE            = None          # ✅ NO rescale — EfficientNet handles it internally

KAGGLE_TRAIN_DIR   = "/kaggle/input/brain-tumor-mri-dataset/Training/"
KAGGLE_TEST_DIR    = "/kaggle/input/datasets/bhavishgupta/newtestdataset/Test"
LOCAL_TRAIN_DIR    = "/content/brain-tumor-mri-dataset/Training/"
LOCAL_TEST_DIR     = "/content/newtestdataset/Test"


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def set_seeds(seed=SEED):
    """Set all random seeds for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def resolve_paths(train_dir=None, test_dir=None):
    if train_dir is None:
        train_dir = KAGGLE_TRAIN_DIR if os.path.exists(KAGGLE_TRAIN_DIR) else LOCAL_TRAIN_DIR
    if test_dir is None:
        test_dir  = KAGGLE_TEST_DIR  if os.path.exists(KAGGLE_TEST_DIR)  else LOCAL_TEST_DIR
    return train_dir, test_dir


# ─────────────────────────────────────────────
# IMAGE DATA GENERATORS
# ─────────────────────────────────────────────

def get_train_datagen():
    """Training-set generator with augmentations EXACTLY as in the paper."""
    return ImageDataGenerator(
        rescale            = RESCALE,
        rotation_range     = ROTATION_RANGE,
        brightness_range   = BRIGHTNESS_RANGE,
        zoom_range         = ZOOM_RANGE,
        width_shift_range  = WIDTH_SHIFT_RANGE,
        height_shift_range = HEIGHT_SHIFT_RANGE,
        fill_mode          = FILL_MODE,
        validation_split   = VALIDATION_SPLIT,
    )


def get_val_test_datagen():
    """Validation / test-set generator — NO augmentation, NO rescale."""
    return ImageDataGenerator(rescale=RESCALE)


# ─────────────────────────────────────────────
# FLOW FROM DIRECTORY
# ─────────────────────────────────────────────

def get_generators(train_dir=None, test_dir=None,
                   batch_size=BATCH_SIZE, img_size=IMG_SIZE, seed=SEED):
    train_dir, test_dir = resolve_paths(train_dir, test_dir)

    train_datagen    = get_train_datagen()
    val_test_datagen = get_val_test_datagen()

    common_kwargs = dict(
        target_size   = img_size,
        batch_size    = batch_size,
        class_mode    = 'categorical',
        classes       = CLASS_NAMES,
        interpolation = 'bilinear',
        seed          = seed,
    )

    train_gen = train_datagen.flow_from_directory(
        train_dir, subset='training', shuffle=True, **common_kwargs)

    val_gen = train_datagen.flow_from_directory(
        train_dir, subset='validation', shuffle=False, **common_kwargs)

    test_gen = val_test_datagen.flow_from_directory(
        test_dir, shuffle=False, **common_kwargs)

    _print_split_summary(train_gen, val_gen, test_gen)
    return train_gen, val_gen, test_gen


def get_test_generator(test_dir=None, batch_size=BATCH_SIZE, img_size=IMG_SIZE):
    _, test_dir = resolve_paths(test_dir=test_dir)
    datagen  = get_val_test_datagen()
    test_gen = datagen.flow_from_directory(
        test_dir,
        target_size   = img_size,
        batch_size    = batch_size,
        class_mode    = 'categorical',
        classes       = CLASS_NAMES,
        interpolation = 'bilinear',
        shuffle       = False
    )
    return test_gen


def _print_split_summary(train_gen, val_gen, test_gen):
    print("\n" + "=" * 55)
    print("  📊 DATASET SPLIT SUMMARY")
    print("=" * 55)
    print(f"  Train samples      : {train_gen.samples}")
    print(f"  Validation samples : {val_gen.samples}")
    print(f"  Test samples       : {test_gen.samples}")
    print(f"  Classes            : {CLASS_NAMES}")
    print(f"  Class indices      : {train_gen.class_indices}")
    print(f"  Image size         : {IMG_SIZE}")
    print(f"  Batch size         : {train_gen.batch_size}")
    print("=" * 55 + "\n")


def print_class_distribution(generator, label="Set"):
    labels = generator.classes
    print(f"\n  📂 {label} class distribution:")
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        count = np.sum(labels == cls_idx)
        print(f"      {cls_name:12s}: {count} images")

In [10]:
# =============================================================
# 🧠 VGG16 — TRANSFER LEARNING MODEL
# Target: 98.78% overall accuracy (early-stop at 48 epochs)
# =============================================================

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

# ✅ FIXED: Replaced "from ..config import ..." with inline values
INPUT_SHAPE     = (240, 240, 3)
NUM_CLASSES     = 4
LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.5
FINE_TUNE_LR    = 1e-5
L2_REG          = 1e-4
LABEL_SMOOTHING = 0.1


def build_vgg16_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                      dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                      l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = VGG16(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(256, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs, name='VGG16_BrainTumor')
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    return model


def get_vgg16_fine_tune_model(model, unfreeze_from_layer='block5_conv1',
                               fine_tune_lr=FINE_TUNE_LR,
                               label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('vgg16')
    base_model.trainable = True

    trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable

    model.compile(
        optimizer=Adam(learning_rate=fine_tune_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    print(f"  ✅ VGG16 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model

In [11]:
# =============================================================
# 🧠 ResNet50 — TRANSFER LEARNING MODEL
# Target: 98.25% overall accuracy (early-stop at 61 epochs)
# =============================================================

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

# ✅ FIXED: Replaced "from ..config import ..." with inline values
INPUT_SHAPE     = (240, 240, 3)
NUM_CLASSES     = 4
LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.5
FINE_TUNE_LR    = 1e-5
L2_REG          = 1e-4
LABEL_SMOOTHING = 0.1


def build_resnet50_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                         dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                         l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(256, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs, name='ResNet50_BrainTumor')
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    return model


def get_resnet50_fine_tune_model(model, unfreeze_from_layer='conv5_block1_1_conv',
                                  fine_tune_lr=FINE_TUNE_LR,
                                  label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('resnet50')
    base_model.trainable = True

    trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable

    model.compile(
        optimizer=Adam(learning_rate=fine_tune_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    print(f"  ✅ ResNet50 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model

In [12]:
# =============================================================
# 🧠 EfficientNet-B0 — TRANSFER LEARNING MODEL
# Target: 95.58% overall accuracy (early-stop at 143 epochs)
# =============================================================

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

# ✅ FIXED: Replaced "from ..config import ..." with inline values
INPUT_SHAPE     = (240, 240, 3)
NUM_CLASSES     = 4
LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.5
FINE_TUNE_LR    = 1e-5
L2_REG          = 1e-4
LABEL_SMOOTHING = 0.1

def build_efficientnetb0_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                                dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                                l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    try:
        base = EfficientNetB0(weights='imagenet', include_top=False,
                              input_tensor=inputs, include_preprocessing=False)
    except TypeError:
        base = EfficientNetB0(weights='imagenet', include_top=False,
                              input_tensor=inputs)

    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(256, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs, name='EfficientNetB0_BrainTumor')
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    return model


def get_efficientnetb0_fine_tune_model(model, unfreeze_from_layer='block6a_expand_conv',
                                        fine_tune_lr=FINE_TUNE_LR,
                                        label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('efficientnetb0')
    base_model.trainable = True

    trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable

    model.compile(
        optimizer=Adam(learning_rate=fine_tune_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    print(f"  ✅ EfficientNetB0 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model

In [14]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — TRAINING SCRIPT
# =============================================================

import os, random, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
warnings.filterwarnings('ignore')

# ✅ FIXED: Replaced all relative imports with inline config values
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
SEED                 = 42
BATCH_SIZE           = 32
EPOCHS               = 50
PHASE1_EPOCHS        = 20
EARLY_STOP_MONITOR   = 'val_accuracy'
EARLY_STOP_PATIENCE  = 10
EARLY_STOP_DELTA     = 0.001
RESTORE_BEST_WEIGHTS = True
REDUCE_LR_FACTOR     = 0.3
REDUCE_LR_PATIENCE   = 3
REDUCE_LR_MIN_LR     = 1e-7
OUTPUT_DIR           = '/kaggle/working/outputs/'
MODEL_NAMES          = ['vgg16', 'resnet50', 'efficientnetb0']
PAPER_TARGETS        = {
    'vgg16':          {'overall': 98.78},
    'resnet50':       {'overall': 97.55},
    'efficientnetb0': {'overall': 98.33},
}

# ✅ FIXED: Replaced from .data_preprocessing import
# ─────────────────────────────────────────────
# DATA PREPROCESSING (inline)
# ─────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE           = (240, 240)
INPUT_SHAPE        = (240, 240, 3)
NUM_CLASSES        = 4
CLASS_NAMES        = ['glioma', 'meningioma', 'notumor', 'pituitary']
VALIDATION_SPLIT   = 0.2
ROTATION_RANGE     = 20
BRIGHTNESS_RANGE   = [0.8, 1.2]
ZOOM_RANGE         = 0.2
WIDTH_SHIFT_RANGE  = 0.2
HEIGHT_SHIFT_RANGE = 0.2
FILL_MODE          = 'nearest'
KAGGLE_TRAIN_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
KAGGLE_TEST_DIR    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
#LOCAL_TRAIN_DIR    = '/content/brain-tumor-mri-dataset/Training/'
#LOCAL_TEST_DIR     = '/content/newtestdataset/Test'

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def resolve_paths(train_dir=None, test_dir=None):
    if train_dir is None:
        train_dir = KAGGLE_TRAIN_DIR if os.path.exists(KAGGLE_TRAIN_DIR) else LOCAL_TRAIN_DIR
    if test_dir is None:
        test_dir  = KAGGLE_TEST_DIR  if os.path.exists(KAGGLE_TEST_DIR)  else LOCAL_TEST_DIR
    return train_dir, test_dir

def get_generators(train_dir=None, test_dir=None,
                   batch_size=BATCH_SIZE, img_size=IMG_SIZE, seed=SEED):
    train_dir, test_dir = resolve_paths(train_dir, test_dir)

    train_datagen = ImageDataGenerator(
        rescale            = None,
        rotation_range     = ROTATION_RANGE,
        brightness_range   = BRIGHTNESS_RANGE,
        zoom_range         = ZOOM_RANGE,
        width_shift_range  = WIDTH_SHIFT_RANGE,
        height_shift_range = HEIGHT_SHIFT_RANGE,
        fill_mode          = FILL_MODE,
        validation_split   = VALIDATION_SPLIT,
    )
    val_test_datagen = ImageDataGenerator()

    common_kwargs = dict(
        target_size   = img_size,
        batch_size    = batch_size,
        class_mode    = 'categorical',
        classes       = CLASS_NAMES,
        interpolation = 'bilinear',
        seed          = seed,
    )

    train_gen = train_datagen.flow_from_directory(
        train_dir, subset='training',   shuffle=True,  **common_kwargs)
    val_gen   = train_datagen.flow_from_directory(
        train_dir, subset='validation', shuffle=False, **common_kwargs)
    test_gen  = val_test_datagen.flow_from_directory(
        test_dir,  shuffle=False, **common_kwargs)

    print("\n" + "="*55)
    print("  📊 DATASET SPLIT SUMMARY")
    print("="*55)
    print(f"  Train samples      : {train_gen.samples}")
    print(f"  Validation samples : {val_gen.samples}")
    print(f"  Test samples       : {test_gen.samples}")
    print(f"  Classes            : {CLASS_NAMES}")
    print(f"  Class indices      : {train_gen.class_indices}")
    print("="*55 + "\n")
    return train_gen, val_gen, test_gen


# ✅ FIXED: Replaced from .models.vgg16_model import
# ─────────────────────────────────────────────
# VGG16 MODEL (inline)
# ─────────────────────────────────────────────
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.5
FINE_TUNE_LR    = 1e-5
L2_REG          = 1e-4
LABEL_SMOOTHING = 0.1

def build_vgg16_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                      dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                      l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = VGG16(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='VGG16_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_vgg16_fine_tune_model(model, unfreeze_from_layer='block5_conv1',
                               fine_tune_lr=FINE_TUNE_LR,
                               label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('vgg16')
    base_model.trainable = True
    trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable
    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ VGG16 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model


# ✅ FIXED: Replaced from .models.resnet50_model import
# ─────────────────────────────────────────────
# RESNET50 MODEL (inline)
# ─────────────────────────────────────────────
from tensorflow.keras.applications import ResNet50

def build_resnet50_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                         dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                         l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='ResNet50_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_resnet50_fine_tune_model(model, unfreeze_from_layer='conv5_block1_1_conv',
                                  fine_tune_lr=FINE_TUNE_LR,
                                  label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('resnet50')
    base_model.trainable = True
    trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable
    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ ResNet50 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model


# ✅ FIXED: Replaced from .models.efficientnet_model import
# ─────────────────────────────────────────────
# EFFICIENTNETB0 MODEL (inline)
# ─────────────────────────────────────────────
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import BatchNormalization

def build_efficientnetb0_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                                dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                                l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='EfficientNetB0_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_efficientnetb0_fine_tune_model(model, unfreeze_last_n=20,
                                        fine_tune_lr=FINE_TUNE_LR,
                                        label_smoothing=LABEL_SMOOTHING):
    base_model = model.get_layer('efficientnetb0')
    base_model.trainable = True
    for layer in base_model.layers[:-unfreeze_last_n]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ EfficientNetB0 fine-tune: last {unfreeze_last_n} layers unfrozen, lr={fine_tune_lr}")
    return model


# ─────────────────────────────────────────────
# CALLBACKS
# ─────────────────────────────────────────────
def make_callbacks(model_name, output_dir=OUTPUT_DIR, phase=1):
    os.makedirs(output_dir, exist_ok=True)
    ckpt_path = os.path.join(output_dir, f"{model_name}_phase{phase}_best.keras")

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path, monitor=EARLY_STOP_MONITOR,
        save_best_only=True, verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor=EARLY_STOP_MONITOR, patience=EARLY_STOP_PATIENCE,
        min_delta=EARLY_STOP_DELTA,
        restore_best_weights=RESTORE_BEST_WEIGHTS, verbose=1)

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor=EARLY_STOP_MONITOR, factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=REDUCE_LR_MIN_LR, verbose=1)

    return [checkpoint, early_stop, reduce_lr]


# ─────────────────────────────────────────────
# TRAINING CURVES
# ─────────────────────────────────────────────
def plot_training_curves(history_phase1, history_phase2,
                          model_name, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    def _concat(key):
        h1 = history_phase1.history.get(key, [])
        h2 = history_phase2.history.get(key, []) if history_phase2 else []
        return h1 + h2

    acc,  val_acc  = _concat('accuracy'),  _concat('val_accuracy')
    loss, val_loss = _concat('loss'),      _concat('val_loss')
    epochs_range   = range(1, len(acc) + 1)
    phase1_end     = len(history_phase1.history.get('accuracy', []))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name.upper()} — Training Curves',
                 fontsize=14, y=1.01)

    axes[0].plot(epochs_range, acc,     label='Train Accuracy',      linewidth=2)
    axes[0].plot(epochs_range, val_acc, label='Validation Accuracy', linewidth=2)
    if history_phase2 and phase1_end > 0:
        axes[0].axvline(phase1_end, color='grey',
                        linestyle='--', label='Fine-tune starts')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(epochs_range, loss,     label='Train Loss',      linewidth=2)
    axes[1].plot(epochs_range, val_loss, label='Validation Loss', linewidth=2)
    if history_phase2 and phase1_end > 0:
        axes[1].axvline(phase1_end, color='grey',
                        linestyle='--', label='Fine-tune starts')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.4)

    plt.tight_layout()
    save_path = os.path.join(output_dir, f"{model_name}_training_curves.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Training curves saved: {save_path}")


# ─────────────────────────────────────────────
# TRAIN SINGLE MODEL
# ─────────────────────────────────────────────
def train_model(model_name, train_gen, val_gen,
                output_dir=OUTPUT_DIR,
                phase1_epochs=PHASE1_EPOCHS,
                total_epochs=EPOCHS):
    print("\n" + "="*60)
    print(f"  🚀 Training: {model_name.upper()}")
    print("="*60)

    build_fn = {
        'vgg16':          build_vgg16_model,
        'resnet50':       build_resnet50_model,
        'efficientnetb0': build_efficientnetb0_model,
    }[model_name]
    model = build_fn()
    model.summary(line_length=90)

    # Phase 1: frozen base
    print(f"\n  📌 Phase 1: training head "
          f"(frozen base, max {phase1_epochs} epochs) …")
    cb_p1      = make_callbacks(model_name, output_dir, phase=1)
    history_p1 = model.fit(train_gen, epochs=phase1_epochs,
                           validation_data=val_gen,
                           callbacks=cb_p1, verbose=1)

    # Phase 2: fine-tuning
    fine_tune_epochs = total_epochs - len(history_p1.history['accuracy'])
    print(f"\n  📌 Phase 2: fine-tuning "
          f"(max {fine_tune_epochs} more epochs) …")

    fine_tune_fn = {
        'vgg16':          get_vgg16_fine_tune_model,
        'resnet50':       get_resnet50_fine_tune_model,
        'efficientnetb0': get_efficientnetb0_fine_tune_model,
    }[model_name]
    model = fine_tune_fn(model)

    cb_p2      = make_callbacks(model_name, output_dir, phase=2)
    history_p2 = model.fit(train_gen, epochs=fine_tune_epochs,
                           validation_data=val_gen,
                           callbacks=cb_p2, verbose=1)

    # Save final model
    save_path = os.path.join(output_dir, f"{model_name}_final.keras")
    model.save(save_path)
    print(f"\n  💾 Model saved: {save_path}")

    plot_training_curves(history_p1, history_p2, model_name, output_dir)

    best_val_acc     = max(history_p1.history['val_accuracy'] +
                           history_p2.history.get('val_accuracy', []))
    paper_target     = PAPER_TARGETS[model_name]['overall']
    total_epochs_run = (len(history_p1.history['accuracy']) +
                        len(history_p2.history.get('accuracy', [])))

    print(f"\n  ✅ {model_name.upper()} complete:")
    print(f"      Total epochs     : {total_epochs_run}")
    print(f"      Best val accuracy: {best_val_acc * 100:.2f}%")
    print(f"      Paper target     : {paper_target:.2f}%")

    return model, history_p1, history_p2


# ─────────────��───────────────────────────────
# TRAIN ALL MODELS
# ─────────────────────────────────────────────
def train_all(train_dir=None, test_dir=None,
              output_dir=OUTPUT_DIR, models_to_train=None):
    set_seeds(SEED)
    if models_to_train is None:
        models_to_train = MODEL_NAMES

    train_gen, val_gen, _ = get_generators(
        train_dir=train_dir, test_dir=test_dir,
        batch_size=BATCH_SIZE, seed=SEED)

    results = {}
    for model_name in models_to_train:
        model, h1, h2 = train_model(
            model_name, train_gen, val_gen, output_dir)
        results[model_name] = (model, h1, h2)

    print("\n" + "="*60)
    print("  🎉 All models trained successfully!")
    print("="*60)
    return results


# ✅ FIXED: Replaced if __name__ == '__main__' + argparse
# with direct call — works in Kaggle notebook
# ─────────────────────────────────────────────
# RUN TRAINING — edit models_to_train as needed
# ─────────────────────────────────────────────
results = train_all(
    train_dir       = None,          # auto-detects Kaggle path
    test_dir        = None,          # auto-detects Kaggle path
    output_dir      = OUTPUT_DIR,
    models_to_train = MODEL_NAMES    # ['vgg16', 'resnet50', 'efficientnetb0']
                                     # or e.g. ['vgg16'] to train one only
)

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

  📊 DATASET SPLIT SUMMARY
  Train samples      : 4480
  Validation samples : 1120
  Test samples       : 1600
  Classes            : ['glioma', 'meningioma', 'notumor', 'pituitary']
  Class indices      : {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


  🚀 Training: VGG16


Model: "VGG16_BrainTumor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)            │ (None, 240, 240, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                 │ (None, 240, 240, 64)         │           1,792 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                 │ (None, 240, 240, 64)         │          36,928 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)            │ (None, 120, 120, 64)         │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                 │ (None, 120, 120, 128)        │          73,856 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                 │ (None, 120, 120, 128)        │         147,584 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)            │ (None, 60, 60, 128)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                 │ (None, 60, 60, 256)          │         295,168 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                 │ (None, 60, 60, 256)          │         590,080 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                 │ (None, 60, 60, 256)          │         590,080 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)            │ (None, 30, 30, 256)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                 │ (None, 30, 30, 512)          │       1,180,160 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                 │ (None, 30, 30, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                 │ (None, 30, 30, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)            │ (None, 15, 15, 512)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)            │ (None, 7, 7, 512)            │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ gap (GlobalAveragePooling2D)          │ (None, 512)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ fc_256 (Dense)                        │ (None, 256)                  │         131,32

 Total params: 14,847,044 (56.64 MB)

 Trainable params: 132,356 (517.02 KB)

...

I0000 00:00:1773162597.992949     199 service.cc:152] XLA service 0x7f224400dff0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773162597.992984     199 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773162597.992991     199 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773162598.643305     199 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/140 ━━━━━━━━━━━━━━━━━━━━ 46:07 20s/step - accuracy: 0.2188 - loss: 7.1992

I0000 00:00:1773162615.742339     199 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 680ms/step - accuracy: 0.5915 - loss: 2.0301
Epoch 1: val_accuracy improved from -inf to 0.83750, saving model to /kaggle/working/outputs/vgg16_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 142s 875ms/step - accuracy: 0.5922 - loss: 2.0248 - val_accuracy: 0.8375 - val_loss: 0.7366 - learning_rate: 0.0010
Epoch 2/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.7887 - loss: 0.8329
Epoch 2: val_accuracy improved from 0.83750 to 0.86786, saving model to /kaggle/working/outputs/vgg16_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 97s 694ms/step - accuracy: 0.7888 - loss: 0.8328 - val_accuracy: 0.8679 - val_loss: 0.6912 - learning_rate: 0.0010
Epoch 3/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 544ms/step - accuracy: 0.8216 - loss: 0.7650
Epoch 3: val_accuracy did not improve from 0.86786
140/140 ━━━━━━━━━━━━━━━━━━━━ 95s 680ms/step - accuracy: 0.8216 - loss: 0.7649 - val_accuracy: 0.8554 - val_loss: 0.6915 - learning_rate: 0.0010
Epoch 4/20
140/140 ━━━━━━━

ValueError: No such layer: vgg16. Existing layers are: ['input_layer_1', 'block1_conv1', 'block1_conv2', 'block1_pool', 'block2_conv1', 'block2_conv2', 'block2_pool', 'block3_conv1', 'block3_conv2', 'block3_conv3', 'block3_pool', 'block4_conv1', 'block4_conv2', 'block4_conv3', 'block4_pool', 'block5_conv1', 'block5_conv2', 'block5_conv3', 'block5_pool', 'gap', 'fc_256', 'dropout', 'predictions'].

In [15]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — TRAINING SCRIPT
# =============================================================

import os, random, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
SEED                 = 42
BATCH_SIZE           = 32
EPOCHS               = 50
PHASE1_EPOCHS        = 20
EARLY_STOP_MONITOR   = 'val_accuracy'
EARLY_STOP_PATIENCE  = 10
EARLY_STOP_DELTA     = 0.001
RESTORE_BEST_WEIGHTS = True
REDUCE_LR_FACTOR     = 0.3
REDUCE_LR_PATIENCE   = 3
REDUCE_LR_MIN_LR     = 1e-7
OUTPUT_DIR           = '/kaggle/working/outputs/'
MODEL_NAMES          = ['vgg16', 'resnet50', 'efficientnetb0']
PAPER_TARGETS        = {
    'vgg16':          {'overall': 98.78},
    'resnet50':       {'overall': 97.55},
    'efficientnetb0': {'overall': 98.33},
}

# ─────────────────────────────────────────────
# DATA PREPROCESSING
# ─────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE           = (240, 240)
INPUT_SHAPE        = (240, 240, 3)
NUM_CLASSES        = 4
CLASS_NAMES        = ['glioma', 'meningioma', 'notumor', 'pituitary']
VALIDATION_SPLIT   = 0.2
ROTATION_RANGE     = 20
BRIGHTNESS_RANGE   = [0.8, 1.2]
ZOOM_RANGE         = 0.2
WIDTH_SHIFT_RANGE  = 0.2
HEIGHT_SHIFT_RANGE = 0.2
FILL_MODE          = 'nearest'
KAGGLE_TRAIN_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
KAGGLE_TEST_DIR    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
#LOCAL_TRAIN_DIR    = '/content/brain-tumor-mri-dataset/Training/'
#LOCAL_TEST_DIR     = '/content/newtestdataset/Test'

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def resolve_paths(train_dir=None, test_dir=None):
    if train_dir is None:
        train_dir = KAGGLE_TRAIN_DIR if os.path.exists(KAGGLE_TRAIN_DIR) else LOCAL_TRAIN_DIR
    if test_dir is None:
        test_dir  = KAGGLE_TEST_DIR  if os.path.exists(KAGGLE_TEST_DIR)  else LOCAL_TEST_DIR
    return train_dir, test_dir

def get_generators(train_dir=None, test_dir=None,
                   batch_size=BATCH_SIZE, img_size=IMG_SIZE, seed=SEED):
    train_dir, test_dir = resolve_paths(train_dir, test_dir)

    train_datagen    = ImageDataGenerator(
        rescale            = None,
        rotation_range     = ROTATION_RANGE,
        brightness_range   = BRIGHTNESS_RANGE,
        zoom_range         = ZOOM_RANGE,
        width_shift_range  = WIDTH_SHIFT_RANGE,
        height_shift_range = HEIGHT_SHIFT_RANGE,
        fill_mode          = FILL_MODE,
        validation_split   = VALIDATION_SPLIT,
    )
    val_test_datagen = ImageDataGenerator()

    common_kwargs = dict(
        target_size   = img_size,
        batch_size    = batch_size,
        class_mode    = 'categorical',
        classes       = CLASS_NAMES,
        interpolation = 'bilinear',
        seed          = seed,
    )

    train_gen = train_datagen.flow_from_directory(
        train_dir, subset='training',   shuffle=True,  **common_kwargs)
    val_gen   = train_datagen.flow_from_directory(
        train_dir, subset='validation', shuffle=False, **common_kwargs)
    test_gen  = val_test_datagen.flow_from_directory(
        test_dir,  shuffle=False, **common_kwargs)

    print("\n" + "="*55)
    print("  📊 DATASET SPLIT SUMMARY")
    print("="*55)
    print(f"  Train samples      : {train_gen.samples}")
    print(f"  Validation samples : {val_gen.samples}")
    print(f"  Test samples       : {test_gen.samples}")
    print(f"  Classes            : {CLASS_NAMES}")
    print(f"  Class indices      : {train_gen.class_indices}")
    print("="*55 + "\n")
    return train_gen, val_gen, test_gen


# ─────────────────────────────────────────────
# MODEL IMPORTS
# ─────────────────────────────────────────────
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, GlobalAveragePooling2D,
                                     Input, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.5
FINE_TUNE_LR    = 1e-5
L2_REG          = 1e-4
LABEL_SMOOTHING = 0.1


# ─────────────────────────────────────────────
# VGG16
# ─────────────────────────────────────────────
def build_vgg16_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                      dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                      l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = VGG16(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='VGG16_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_vgg16_fine_tune_model(model, unfreeze_from_layer='block5_conv1',
                               fine_tune_lr=FINE_TUNE_LR,
                               label_smoothing=LABEL_SMOOTHING):
    # ✅ FIXED: VGG16 layers are directly in model, not wrapped
    # iterate model.layers directly instead of model.get_layer('vgg16')
    trainable = False
    for layer in model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable
    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ VGG16 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model


# ─────────────────────────────────────────────
# RESNET50
# ─────────────────────────────────────────────
def build_resnet50_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                         dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                         l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='ResNet50_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_resnet50_fine_tune_model(model, unfreeze_from_layer='conv5_block1_1_conv',
                                  fine_tune_lr=FINE_TUNE_LR,
                                  label_smoothing=LABEL_SMOOTHING):
    # ✅ FIXED: iterate model.layers directly
    trainable = False
    for layer in model.layers:
        if layer.name == unfreeze_from_layer:
            trainable = True
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable
    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ ResNet50 fine-tune: layers from '{unfreeze_from_layer}' unfrozen, lr={fine_tune_lr}")
    return model


# ─────────────────────────────────────────────
# EFFICIENTNETB0
# ─────────────────────────────────────────────
def build_efficientnetb0_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
                                dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
                                l2_reg=L2_REG, label_smoothing=LABEL_SMOOTHING):
    inputs = Input(shape=input_shape)
    base   = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inputs)
    base.trainable = False
    x       = base.output
    x       = GlobalAveragePooling2D(name='gap')(x)
    x       = Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_reg), name='fc_256')(x)
    x       = Dropout(dropout_rate, name='dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    model   = Model(inputs=inputs, outputs=outputs, name='EfficientNetB0_BrainTumor')
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    return model

def get_efficientnetb0_fine_tune_model(model, unfreeze_last_n=20,
                                        fine_tune_lr=FINE_TUNE_LR,
                                        label_smoothing=LABEL_SMOOTHING):
    # ✅ FIXED: iterate model.layers directly, unfreeze last N non-BN layers
    non_bn_layers = [l for l in model.layers
                     if not isinstance(l, tf.keras.layers.BatchNormalization)]
    freeze_until  = len(non_bn_layers) - unfreeze_last_n

    for i, layer in enumerate(model.layers):
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            idx = non_bn_layers.index(layer)
            layer.trainable = (idx >= freeze_until)

    model.compile(
        optimizer = Adam(learning_rate=fine_tune_lr),
        loss      = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics   = ['accuracy'],
    )
    print(f"  ✅ EfficientNetB0 fine-tune: last {unfreeze_last_n} layers unfrozen, lr={fine_tune_lr}")
    return model


# ─────────────────────────────────────────────
# CALLBACKS
# ─────────────────────────────────────────────
def make_callbacks(model_name, output_dir=OUTPUT_DIR, phase=1):
    os.makedirs(output_dir, exist_ok=True)
    ckpt_path = os.path.join(output_dir, f"{model_name}_phase{phase}_best.keras")

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path, monitor=EARLY_STOP_MONITOR,
        save_best_only=True, verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor=EARLY_STOP_MONITOR, patience=EARLY_STOP_PATIENCE,
        min_delta=EARLY_STOP_DELTA,
        restore_best_weights=RESTORE_BEST_WEIGHTS, verbose=1)

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor=EARLY_STOP_MONITOR, factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=REDUCE_LR_MIN_LR, verbose=1)

    return [checkpoint, early_stop, reduce_lr]


# ─────────────────────────────────────────────
# TRAINING CURVES
# ─────────────────────────────────────────────
def plot_training_curves(history_phase1, history_phase2,
                          model_name, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    def _concat(key):
        h1 = history_phase1.history.get(key, [])
        h2 = history_phase2.history.get(key, []) if history_phase2 else []
        return h1 + h2

    acc,  val_acc  = _concat('accuracy'),  _concat('val_accuracy')
    loss, val_loss = _concat('loss'),      _concat('val_loss')
    epochs_range   = range(1, len(acc) + 1)
    phase1_end     = len(history_phase1.history.get('accuracy', []))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name.upper()} — Training Curves',
                 fontsize=14, y=1.01)

    axes[0].plot(epochs_range, acc,     label='Train Accuracy',      linewidth=2)
    axes[0].plot(epochs_range, val_acc, label='Validation Accuracy', linewidth=2)
    if history_phase2 and phase1_end > 0:
        axes[0].axvline(phase1_end, color='grey',
                        linestyle='--', label='Fine-tune starts')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(epochs_range, loss,     label='Train Loss',      linewidth=2)
    axes[1].plot(epochs_range, val_loss, label='Validation Loss', linewidth=2)
    if history_phase2 and phase1_end > 0:
        axes[1].axvline(phase1_end, color='grey',
                        linestyle='--', label='Fine-tune starts')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.4)

    plt.tight_layout()
    save_path = os.path.join(output_dir, f"{model_name}_training_curves.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Training curves saved: {save_path}")


# ─────────────────────────────────────────────
# TRAIN SINGLE MODEL
# ─────────────────────────────────────────────
def train_model(model_name, train_gen, val_gen,
                output_dir=OUTPUT_DIR,
                phase1_epochs=PHASE1_EPOCHS,
                total_epochs=EPOCHS):
    print("\n" + "="*60)
    print(f"  ��� Training: {model_name.upper()}")
    print("="*60)

    build_fn = {
        'vgg16':          build_vgg16_model,
        'resnet50':       build_resnet50_model,
        'efficientnetb0': build_efficientnetb0_model,
    }[model_name]
    model = build_fn()
    model.summary(line_length=90)

    # Phase 1
    print(f"\n  📌 Phase 1: training head "
          f"(frozen base, max {phase1_epochs} epochs) …")
    cb_p1      = make_callbacks(model_name, output_dir, phase=1)
    history_p1 = model.fit(train_gen, epochs=phase1_epochs,
                           validation_data=val_gen,
                           callbacks=cb_p1, verbose=1)

    # Phase 2
    fine_tune_epochs = total_epochs - len(history_p1.history['accuracy'])
    print(f"\n  📌 Phase 2: fine-tuning "
          f"(max {fine_tune_epochs} more epochs) …")

    fine_tune_fn = {
        'vgg16':          get_vgg16_fine_tune_model,
        'resnet50':       get_resnet50_fine_tune_model,
        'efficientnetb0': get_efficientnetb0_fine_tune_model,
    }[model_name]
    model = fine_tune_fn(model)

    cb_p2      = make_callbacks(model_name, output_dir, phase=2)
    history_p2 = model.fit(train_gen, epochs=fine_tune_epochs,
                           validation_data=val_gen,
                           callbacks=cb_p2, verbose=1)

    # Save
    save_path = os.path.join(output_dir, f"{model_name}_final.keras")
    model.save(save_path)
    print(f"\n  💾 Model saved: {save_path}")

    plot_training_curves(history_p1, history_p2, model_name, output_dir)

    best_val_acc     = max(history_p1.history['val_accuracy'] +
                           history_p2.history.get('val_accuracy', []))
    paper_target     = PAPER_TARGETS[model_name]['overall']
    total_epochs_run = (len(history_p1.history['accuracy']) +
                        len(history_p2.history.get('accuracy', [])))

    print(f"\n  ✅ {model_name.upper()} complete:")
    print(f"      Total epochs     : {total_epochs_run}")
    print(f"      Best val accuracy: {best_val_acc * 100:.2f}%")
    print(f"      Paper target     : {paper_target:.2f}%")

    return model, history_p1, history_p2


# ─────────────────────────────────────────────
# TRAIN ALL
# ─────────────────────────────────────────────
def train_all(train_dir=None, test_dir=None,
              output_dir=OUTPUT_DIR, models_to_train=None):
    set_seeds(SEED)
    if models_to_train is None:
        models_to_train = MODEL_NAMES

    train_gen, val_gen, _ = get_generators(
        train_dir=train_dir, test_dir=test_dir,
        batch_size=BATCH_SIZE, seed=SEED)

    results = {}
    for model_name in models_to_train:
        model, h1, h2 = train_model(
            model_name, train_gen, val_gen, output_dir)
        results[model_name] = (model, h1, h2)

    print("\n" + "="*60)
    print("  🎉 All models trained successfully!")
    print("="*60)
    return results


# ─────────────────────────────────────────────
# RUN — direct call for Kaggle notebook
# ─────────────────────────────────────────────
results = train_all(
    train_dir       = None,
    test_dir        = None,
    output_dir      = OUTPUT_DIR,
    models_to_train = MODEL_NAMES   # change to e.g. ['vgg16'] to train one only
)

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

  📊 DATASET SPLIT SUMMARY
  Train samples      : 4480
  Validation samples : 1120
  Test samples       : 1600
  Classes            : ['glioma', 'meningioma', 'notumor', 'pituitary']
  Class indices      : {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


  ��� Training: VGG16


Model: "VGG16_BrainTumor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)            │ (None, 240, 240, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                 │ (None, 240, 240, 64)         │           1,792 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                 │ (None, 240, 240, 64)         │          36,928 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)            │ (None, 120, 120, 64)         │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                 │ (None, 120, 120, 128)        │          73,856 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                 │ (None, 120, 120, 128)        │         147,584 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)            │ (None, 60, 60, 128)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                 │ (None, 60, 60, 256)          │         295,168 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                 │ (None, 60, 60, 256)          │         590,080 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                 │ (None, 60, 60, 256)          │         590,080 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)            │ (None, 30, 30, 256)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                 │ (None, 30, 30, 512)          │       1,180,160 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                 │ (None, 30, 30, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                 │ (None, 30, 30, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)            │ (None, 15, 15, 512)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                 │ (None, 15, 15, 512)          │       2,359,808 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)            │ (None, 7, 7, 512)            │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ gap (GlobalAveragePooling2D)          │ (None, 512)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ fc_256 (Dense)                        │ (None, 256)                  │         131,32

 Total params: 14,847,044 (56.64 MB)

 Trainable params: 132,356 (517.02 KB)

...

 Trainable params: 328,964 (1.25 MB)

 Non-trainable params: 4,049,571 (15.45 MB)


  📌 Phase 1: training head (frozen base, max 20 epochs) …
Epoch 1/20


2026-03-10 20:12:40.296809: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-10 20:12:40.441452: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-10 20:12:40.797205: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-10 20:12:40.938165: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-10 20:12:41.647752: E external/local_xla/xla/stream_

140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 513ms/step - accuracy: 0.6921 - loss: 0.9825
Epoch 1: val_accuracy improved from -inf to 0.85625, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 123s 700ms/step - accuracy: 0.6926 - loss: 0.9816 - val_accuracy: 0.8562 - val_loss: 0.6929 - learning_rate: 0.0010
Epoch 2/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - accuracy: 0.8323 - loss: 0.7352
Epoch 2: val_accuracy improved from 0.85625 to 0.87768, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 91s 647ms/step - accuracy: 0.8324 - loss: 0.7351 - val_accuracy: 0.8777 - val_loss: 0.6624 - learning_rate: 0.0010
Epoch 3/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 517ms/step - accuracy: 0.8495 - loss: 0.6913
Epoch 3: val_accuracy improved from 0.87768 to 0.88304, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 92s 656ms/step - accuracy: 0.8495 - loss: 0.6

In [1]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — EVALUATION SCRIPT
# =============================================================

import os, argparse, warnings, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)
from scipy.stats import kruskal

try:
    from .config import CLASS_NAMES, NUM_CLASSES, OUTPUT_DIR, BATCH_SIZE, MODEL_NAMES, PAPER_TARGETS
    from .data_preprocessing import get_test_generator
except ImportError:
    _pkg_dir = os.path.dirname(os.path.abspath(__file__))
    _parent = os.path.dirname(_pkg_dir)
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from paper_replication.config import CLASS_NAMES, NUM_CLASSES, OUTPUT_DIR, BATCH_SIZE, MODEL_NAMES, PAPER_TARGETS
    from paper_replication.data_preprocessing import get_test_generator


def predict(model, test_gen, batch_size=BATCH_SIZE):
    test_gen.reset()
    steps = int(np.ceil(test_gen.samples / batch_size))
    probs = model.predict(test_gen, steps=steps, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_gen.classes[:len(y_pred)]
    return y_true, y_pred


def per_class_accuracy(y_true, y_pred, class_names=CLASS_NAMES):
    accs = {}
    for idx, name in enumerate(class_names):
        mask = y_true == idx
        if mask.sum() == 0:
            accs[name] = float('nan')
        else:
            accs[name] = accuracy_score(y_true[mask], y_pred[mask])
    return accs


def plot_confusion_matrix(y_true, y_pred, model_name, output_dir=OUTPUT_DIR, class_names=CLASS_NAMES):
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='lightgrey')
    ax.set_title(f'{model_name.upper()} — Confusion Matrix', fontsize=13)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{model_name}_confusion_matrix.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Confusion matrix saved: {save_path}")


def evaluate_model(model, test_gen, model_name, output_dir=OUTPUT_DIR):
    print(f"\n  🔍 Evaluating: {model_name.upper()} …")
    y_true, y_pred = predict(model, test_gen)

    overall_acc = accuracy_score(y_true, y_pred)
    cls_accs = per_class_accuracy(y_true, y_pred)

    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    paper_target = PAPER_TARGETS.get(model_name, {}).get('overall', 'N/A')
    print(f"\n  ─── {model_name.upper()} Results ───")
    print(f"  Overall Accuracy : {overall_acc * 100:.2f}%  (paper target: {paper_target}%)")
    print(f"  Macro Precision  : {macro_prec * 100:.2f}%")
    print(f"  Macro Recall     : {macro_rec  * 100:.2f}%")
    print(f"  Macro F1-Score   : {macro_f1   * 100:.2f}%")
    print("\n  Per-Class Accuracy:")
    for cls, acc in cls_accs.items():
        paper_cls = PAPER_TARGETS.get(model_name, {}).get('per_class', {}).get(cls, 'N/A')
        print(f"      {cls:12s}: {acc * 100:6.2f}%  (paper: {paper_cls}%)")

    print("\n  Full Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    plot_confusion_matrix(y_true, y_pred, model_name, output_dir)

    return {
        'model_name': model_name, 'overall_acc': overall_acc, 'cls_accs': cls_accs,
        'macro_precision': macro_prec, 'macro_recall': macro_rec, 'macro_f1': macro_f1,
        'y_true': y_true, 'y_pred': y_pred,
    }


def plot_model_comparison(metrics_list, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    model_names  = [m['model_name'].upper() for m in metrics_list]
    overall_accs = [m['overall_acc'] * 100 for m in metrics_list]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Model Comparison — Brain Tumor Classification', fontsize=14, y=1.01)

    bars = axes[0].bar(model_names, overall_accs,
                       color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white')
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=10)
    axes[0].set_title('Overall Accuracy (%)'); axes[0].set_ylim(85, 102)
    axes[0].set_ylabel('Accuracy (%)'); axes[0].grid(axis='y', alpha=0.4)

    x = np.arange(len(CLASS_NAMES))
    bar_width = 0.22
    colours = ['#4C72B0', '#DD8452', '#55A868']
    for i, m in enumerate(metrics_list):
        accs = [m['cls_accs'].get(c, 0) * 100 for c in CLASS_NAMES]
        offset = (i - len(metrics_list) / 2 + 0.5) * bar_width
        b = axes[1].bar(x + offset, accs, bar_width, label=model_names[i], color=colours[i])
        axes[1].bar_label(b, fmt='%.1f', padding=2, fontsize=7)

    axes[1].set_title('Per-Class Accuracy (%)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c.capitalize() for c in CLASS_NAMES])
    axes[1].set_ylim(75, 108); axes[1].legend(); axes[1].grid(axis='y', alpha=0.4)

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'model_comparison.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n  💾 Model comparison chart saved: {save_path}")


def kruskal_wallis_test(metrics_list):
    groups = [list(m['cls_accs'].values()) for m in metrics_list]
    if len(groups) < 2:
        print("  ⚠️  Need ≥2 models for Kruskal-Wallis test.")
        return {}
    h_stat, p_val = kruskal(*groups)
    sig = '(significant at α=0.05)' if p_val < 0.05 else '(not significant)'
    print(f"\n  📊 Kruskal-Wallis Test: H = {h_stat:.4f}, p = {p_val:.4f}  {sig}")
    print(f"     Paper reference   : H = 7.88,   p = 0.019")
    return {'statistic': h_stat, 'pvalue': p_val}


def evaluate_all(model_paths, test_dir=None, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    test_gen = get_test_generator(test_dir=test_dir)
    all_metrics = []

    for model_name, model_path in model_paths.items():
        if not os.path.exists(model_path):
            print(f"  ❌ Model file not found, skipping: {model_path}")
            continue
        print(f"\n  📂 Loading model: {model_path}")
        model = tf.keras.models.load_model(model_path)
        metrics = evaluate_model(model, test_gen, model_name, output_dir)
        all_metrics.append(metrics)

    if len(all_metrics) >= 2:
        plot_model_comparison(all_metrics, output_dir)
        kruskal_wallis_test(all_metrics)

    return all_metrics


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Evaluate brain tumor models')
    parser.add_argument('--test_dir', default=None)
    parser.add_argument('--output_dir', default=OUTPUT_DIR)
    parser.add_argument('--vgg16', default=None)
    parser.add_argument('--resnet50', default=None)
    parser.add_argument('--efficientnetb0', default=None)
    args = parser.parse_args()

    model_paths = {}
    if args.vgg16: model_paths['vgg16'] = args.vgg16
    if args.resnet50: model_paths['resnet50'] = args.resnet50
    if args.efficientnetb0: model_paths['efficientnetb0'] = args.efficientnetb0

    if not model_paths:
        for name in MODEL_NAMES:
            p = os.path.join(args.output_dir, f"{name}_final.keras")
            if os.path.exists(p): model_paths[name] = p

    if not model_paths:
        print("  ❌ No model paths found. Run train.py first.")
    else:
        evaluate_all(model_paths, args.test_dir, args.output_dir)

2026-03-11 05:01:38.082824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773205298.462142      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773205298.578875      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773205299.507188      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773205299.507236      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773205299.507239      55 computation_placer.cc:177] computation placer alr

NameError: name '__file__' is not defined

In [1]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — COMPLETE KAGGLE NOTEBOOK
# =============================================================
# This single file contains EVERYTHING:
#   1. Configuration
#   2. Data Preprocessing
#   3. Model Building (VGG16, ResNet50, EfficientNetB0)
#   4. Training with Callbacks
#   5. Evaluation + Confusion Matrix
#   6. Model Comparison + Kruskal-Wallis Test
#   7. Grad-CAM (paper-faithful)
#   8. LIME (paper-faithful with Quickshift)
#
# Instructions:
#   - Upload to Kaggle as a notebook
#   - Attach your brain tumor MRI dataset
#   - Update TRAIN_DIR / TEST_DIR paths if needed
#   - Enable GPU accelerator (Settings → Accelerator → GPU)
#   - Click "Run All"
# =============================================================


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 1 — INSTALL DEPENDENCIES                            ║
# ╚═══════════════════════════════════════════════════════════════╝

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime', '-q'])
print("✅ Dependencies installed.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 2 — IMPORTS                                         ║
# ╚═══════════════════════════════════════════════════════════════╝

import os, glob, warnings
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)
from scipy.stats import kruskal

from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries, quickshift

warnings.filterwarnings('ignore')

print("✅ All imports loaded.")
print(f"   TensorFlow version : {tf.__version__}")
print(f"   GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 3 — CONFIGURATION                                   ║
# ╚═══════════════════════════════════���═══════════════════════════╝

# ── Class / Model Setup ──────────────────────────────────────
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES  = len(CLASS_NAMES)
MODEL_NAMES  = ['vgg16', 'resnet50', 'efficientnetb0']
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 30
LEARNING_RATE = 1e-4

# ── Paths — UPDATE THESE to match your Kaggle dataset ────────
# Click the dataset in Kaggle sidebar to find the exact path
TRAIN_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Paper-reported accuracy targets ──────────────────────────
PAPER_TARGETS = {
    'vgg16': {
        'overall': 96.08,
        'per_class': {
            'glioma': 95.67, 'meningioma': 93.21,
            'notumor': 99.36, 'pituitary': 96.08,
        },
    },
    'resnet50': {
        'overall': 97.06,
        'per_class': {
            'glioma': 97.33, 'meningioma': 94.44,
            'notumor': 99.36, 'pituitary': 97.06,
        },
    },
    'efficientnetb0': {
        'overall': 97.71,
        'per_class': {
            'glioma': 98.00, 'meningioma': 96.30,
            'notumor': 99.36, 'pituitary': 97.71,
        },
    },
}

print("\n✅ Configuration loaded.")
print(f"   Classes      : {CLASS_NAMES}")
print(f"   Image size   : {IMG_SIZE}")
print(f"   Batch size   : {BATCH_SIZE}")
print(f"   Epochs       : {EPOCHS}")
print(f"   Train dir    : {TRAIN_DIR}")
print(f"   Test dir     : {TEST_DIR}")
print(f"   Output dir   : {OUTPUT_DIR}")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 4 — DATA PREPROCESSING                              ║
# ╚═══════════════════════════════════════════════════════════════╝

# ── Augmentation for training ────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2,
)

# ── No augmentation for test / validation ────────────────────
test_datagen = ImageDataGenerator(rescale=1.0 / 255)


def get_train_generators(train_dir=TRAIN_DIR):
    """Returns (train_gen, val_gen) with 80/20 split."""
    train_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES,
        subset='training',
        shuffle=True,
        seed=42,
    )
    val_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES,
        subset='validation',
        shuffle=False,
        seed=42,
    )
    print(f"   Training samples   : {train_gen.samples}")
    print(f"   Validation samples : {val_gen.samples}")
    return train_gen, val_gen


def get_test_generator(test_dir=None):
    """Returns test generator (no augmentation, no shuffle)."""
    if test_dir is None:
        test_dir = TEST_DIR
    test_gen = test_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES,
        shuffle=False,
    )
    print(f"   Test samples       : {test_gen.samples}")
    return test_gen


print("\n✅ Data preprocessing helpers ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 5 — MODEL BUILDING                                  ║
# ╚═══════════════════════════════════════════════════════════════╝

def build_model(model_name, input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    """
    Build a transfer-learning model:
      - Pretrained base (ImageNet, frozen)
      - GlobalAveragePooling2D
      - Dense(256) → BatchNorm → ReLU → Dropout(0.5)
      - Dense(num_classes, softmax)
    """
    base_kwargs = dict(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )

    if model_name == 'vgg16':
        base = VGG16(**base_kwargs)
    elif model_name == 'resnet50':
        base = ResNet50(**base_kwargs)
    elif model_name == 'efficientnetb0':
        base = EfficientNetB0(**base_kwargs)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Freeze base layers
    for layer in base.layers:
        layer.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    total_params = model.count_params()
    trainable    = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
    print(f"\n   📐 {model_name.upper()}")
    print(f"      Total params     : {total_params:,}")
    print(f"      Trainable params : {trainable:,}")
    return model


print("✅ Model builder ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 6 — TRAINING                                        ║
# ╚═══════════════════════════════════════════════════════════════╝

def get_callbacks(model_name, output_dir=OUTPUT_DIR):
    """EarlyStopping + ReduceLR + ModelCheckpoint."""
    return [
        EarlyStopping(
            monitor='val_loss', patience=7,
            restore_best_weights=True, verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-7, verbose=1,
        ),
        ModelCheckpoint(
            os.path.join(output_dir, f'{model_name}_best.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=1,
        ),
    ]


def plot_training_history(history, model_name, output_dir=OUTPUT_DIR):
    """Save accuracy & loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'],  label='Val Accuracy')
    axes[0].set_title(f'{model_name.upper()} — Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'],  label='Val Loss')
    axes[1].set_title(f'{model_name.upper()} — Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(output_dir, f'{model_name}_training_history.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Training history saved: {save_path}")


def train_model(model_name, train_dir=TRAIN_DIR, output_dir=OUTPUT_DIR):
    """Train one model end-to-end and save it."""
    print(f"\n{'='*60}")
    print(f"  🚀 TRAINING: {model_name.upper()}")
    print(f"{'='*60}")

    train_gen, val_gen = get_train_generators(train_dir)
    model = build_model(model_name)
    callbacks = get_callbacks(model_name, output_dir)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    # Save final model
    final_path = os.path.join(output_dir, f'{model_name}_final.keras')
    model.save(final_path)
    print(f"  💾 Final model saved: {final_path}")

    plot_training_history(history, model_name, output_dir)

    return model, history


def train_all(model_names=MODEL_NAMES, train_dir=TRAIN_DIR,
              output_dir=OUTPUT_DIR):
    """Train all models sequentially."""
    os.makedirs(output_dir, exist_ok=True)
    trained = {}
    for name in model_names:
        model, history = train_model(name, train_dir, output_dir)
        trained[name] = {'model': model, 'history': history}
    print(f"\n  🎉 All {len(trained)} models trained!")
    return trained


print("✅ Training functions ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 7 — EVALUATION + CONFUSION MATRIX                   ║
# ╚═══════════════════════════════════════════════════════════════╝

def predict(model, test_gen, batch_size=BATCH_SIZE):
    """Get predictions from model on test generator."""
    test_gen.reset()
    steps = int(np.ceil(test_gen.samples / batch_size))
    probs = model.predict(test_gen, steps=steps, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_gen.classes[:len(y_pred)]
    return y_true, y_pred


def per_class_accuracy(y_true, y_pred, class_names=CLASS_NAMES):
    """Compute accuracy for each class separately."""
    accs = {}
    for idx, name in enumerate(class_names):
        mask = y_true == idx
        if mask.sum() == 0:
            accs[name] = float('nan')
        else:
            accs[name] = accuracy_score(y_true[mask], y_pred[mask])
    return accs


def plot_confusion_matrix(y_true, y_pred, model_name,
                          output_dir=OUTPUT_DIR, class_names=CLASS_NAMES):
    """Generate and save confusion matrix heatmap."""
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='lightgrey')
    ax.set_title(f'{model_name.upper()} — Confusion Matrix', fontsize=13)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{model_name}_confusion_matrix.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Confusion matrix saved: {save_path}")


def evaluate_model(model, test_gen, model_name, output_dir=OUTPUT_DIR):
    """Full evaluation: accuracy, precision, recall, F1, confusion matrix."""
    print(f"\n  🔍 Evaluating: {model_name.upper()} …")
    y_true, y_pred = predict(model, test_gen)

    overall_acc = accuracy_score(y_true, y_pred)
    cls_accs    = per_class_accuracy(y_true, y_pred)

    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    paper_target = PAPER_TARGETS.get(model_name, {}).get('overall', 'N/A')
    print(f"\n  ─── {model_name.upper()} Results ───")
    print(f"  Overall Accuracy : {overall_acc * 100:.2f}%  (paper target: {paper_target}%)")
    print(f"  Macro Precision  : {macro_prec * 100:.2f}%")
    print(f"  Macro Recall     : {macro_rec  * 100:.2f}%")
    print(f"  Macro F1-Score   : {macro_f1   * 100:.2f}%")
    print("\n  Per-Class Accuracy:")
    for cls, acc in cls_accs.items():
        paper_cls = PAPER_TARGETS.get(model_name, {}).get('per_class', {}).get(cls, 'N/A')
        print(f"      {cls:12s}: {acc * 100:6.2f}%  (paper: {paper_cls}%)")

    print("\n  Full Classification Report:")
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, zero_division=0))

    plot_confusion_matrix(y_true, y_pred, model_name, output_dir)

    return {
        'model_name': model_name, 'overall_acc': overall_acc,
        'cls_accs': cls_accs,
        'macro_precision': macro_prec, 'macro_recall': macro_rec,
        'macro_f1': macro_f1,
        'y_true': y_true, 'y_pred': y_pred,
    }


def plot_model_comparison(metrics_list, output_dir=OUTPUT_DIR):
    """Bar chart comparing all models: overall + per-class accuracy."""
    os.makedirs(output_dir, exist_ok=True)
    model_names  = [m['model_name'].upper() for m in metrics_list]
    overall_accs = [m['overall_acc'] * 100   for m in metrics_list]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Model Comparison — Brain Tumor Classification',
                 fontsize=14, y=1.01)

    # Overall accuracy
    bars = axes[0].bar(model_names, overall_accs,
                       color=['#4C72B0', '#DD8452', '#55A868'],
                       edgecolor='white')
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=10)
    axes[0].set_title('Overall Accuracy (%)')
    axes[0].set_ylim(85, 102)
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].grid(axis='y', alpha=0.4)

    # Per-class accuracy
    x = np.arange(len(CLASS_NAMES))
    bar_width = 0.22
    colours = ['#4C72B0', '#DD8452', '#55A868']
    for i, m in enumerate(metrics_list):
        accs   = [m['cls_accs'].get(c, 0) * 100 for c in CLASS_NAMES]
        offset = (i - len(metrics_list) / 2 + 0.5) * bar_width
        b = axes[1].bar(x + offset, accs, bar_width,
                        label=model_names[i], color=colours[i])
        axes[1].bar_label(b, fmt='%.1f', padding=2, fontsize=7)

    axes[1].set_title('Per-Class Accuracy (%)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c.capitalize() for c in CLASS_NAMES])
    axes[1].set_ylim(75, 108)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.4)

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'model_comparison.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n  💾 Model comparison chart saved: {save_path}")


def kruskal_wallis_test(metrics_list):
    """Kruskal-Wallis H-test across model per-class accuracies."""
    groups = [list(m['cls_accs'].values()) for m in metrics_list]
    if len(groups) < 2:
        print("  ⚠️  Need ≥2 models for Kruskal-Wallis test.")
        return {}
    h_stat, p_val = kruskal(*groups)
    sig = '(significant at α=0.05)' if p_val < 0.05 else '(not significant)'
    print(f"\n  📊 Kruskal-Wallis Test: H = {h_stat:.4f}, p = {p_val:.4f}  {sig}")
    print(f"     Paper reference   : H = 7.88,   p = 0.019")
    return {'statistic': h_stat, 'pvalue': p_val}


def evaluate_all(model_paths, test_dir=None, output_dir=OUTPUT_DIR):
    """Evaluate all models, generate comparison chart, run stat test."""
    os.makedirs(output_dir, exist_ok=True)
    test_gen = get_test_generator(test_dir=test_dir)
    all_metrics = []

    for model_name, model_path in model_paths.items():
        if not os.path.exists(model_path):
            print(f"  ❌ Model file not found, skipping: {model_path}")
            continue
        print(f"\n  📂 Loading model: {model_path}")
        model = tf.keras.models.load_model(model_path)
        metrics = evaluate_model(model, test_gen, model_name, output_dir)
        all_metrics.append(metrics)

    if len(all_metrics) >= 2:
        plot_model_comparison(all_metrics, output_dir)
        kruskal_wallis_test(all_metrics)

    return all_metrics


print("✅ Evaluation functions ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 8 — GRAD-CAM (Paper-faithful)                       ║
# ╚═══════════════════════════════════════════════════════════════╝
# Flowchart steps:
#   1. Compute gradient   →  G = ∂yc / ∂Ak           (Eq 18)
#   2. Global avg pooling  →  αck = (1/Z) Σi Σj G     (Eq 19)
#   3. Weight + ReLU      →  Lc = ReLU( Σk αck·Ak )   (Eq 20)
#   4. Upsample + overlay →  bilinear interpolation    (Eq 21)
# ═════════════════════════════════════════════════════════════════

def find_last_conv_layer(model):
    """Walk backwards to find the last Conv2D layer."""
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D,
                              tf.keras.layers.DepthwiseConv2D)):
            return layer.name
    raise ValueError("No Conv2D layer found in model.")


def make_gradcam_heatmap(img_array, model, last_conv_layer_name,
                         pred_index=None):
    """
    Steps 1–3 of the paper's Grad-CAM pipeline.

    (Eq 18)  G = ∂yc / ∂Ak
    (Eq 19)  αck = (1/Z) Σi Σj (∂yc / ∂Ak,ij)
    (Eq 20)  Lc  = ReLU( Σk αck · Ak )

    Returns: heatmap [0,1], pred_idx, pred_conf (%)
    """
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,  # Ak
            model.output,                                   # yc
        ],
    )

    # STEP 1: Compute gradient  G = ∂yc / ∂Ak
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_score = predictions[:, pred_index]

    gradients = tape.gradient(class_score, conv_outputs)       # Eq (18)

    # STEP 2: Global average pooling  αck = (1/Z) Σi Σj G
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))        # Eq (19)

    # STEP 3: Weighted combination + ReLU  Lc = ReLU( Σk αck·Ak )
    conv_outputs = conv_outputs[0]
    cam = conv_outputs @ weights[..., tf.newaxis]
    cam = tf.squeeze(cam)
    cam = tf.nn.relu(cam)                                      # Eq (20)

    # Normalise to [0, 1]
    cam_max = tf.reduce_max(cam)
    if cam_max > 0:
        cam = cam / cam_max

    heatmap   = cam.numpy()
    pred_idx  = int(pred_index)
    pred_conf = float(predictions[0][pred_idx]) * 100
    return heatmap, pred_idx, pred_conf


def overlay_gradcam(img_path, heatmap, alpha=0.4):
    """
    STEP 4: Upsample + Overlay  (Eq 21)
    Lc_upsampled = interpolate(Lc_nor, size=(H, W))
    Uses bilinear interpolation (cv2.INTER_LINEAR).
    """
    original_bgr = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
    H, W = original_rgb.shape[:2]

    # Bilinear interpolation — Eq (21)
    heatmap_upsampled = cv2.resize(heatmap, (W, H),
                                   interpolation=cv2.INTER_LINEAR)

    heatmap_uint8 = np.uint8(255 * heatmap_upsampled)
    jet_coloured  = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    jet_rgb       = cv2.cvtColor(jet_coloured, cv2.COLOR_BGR2RGB)

    superimposed = np.uint8(jet_rgb * alpha + original_rgb * (1 - alpha))
    return superimposed, original_rgb, heatmap_upsampled


def load_image_for_model(img_path, target_size=IMG_SIZE):
    """Load single image, normalise to [0,1], add batch dim."""
    img = keras_image.load_img(img_path, target_size=target_size)
    arr = keras_image.img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)


def run_gradcam_for_model(model, model_name, test_dir=TEST_DIR,
                          output_dir=OUTPUT_DIR, samples_per_class=2):
    """Generate Grad-CAM grid: Original | Activation Map | Overlay."""
    last_conv = find_last_conv_layer(model)
    print(f"\n  🔥 Grad-CAM for {model_name.upper()}  (layer: {last_conv})")

    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 3

    fig = plt.figure(figsize=(5 * n_cols, 5 * n_rows))
    gs  = gridspec.GridSpec(n_rows, n_cols, wspace=0.05, hspace=0.30)

    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = sorted(glob.glob(os.path.join(cls_dir, '*')))[:samples_per_class]

        for s, img_path in enumerate(img_files):
            img_tensor = load_image_for_model(img_path)

            heatmap, pred_idx, pred_conf = make_gradcam_heatmap(
                img_tensor, model, last_conv,
            )
            pred_label = CLASS_NAMES[pred_idx]
            superimposed, original_rgb, heatmap_up = overlay_gradcam(
                img_path, heatmap,
            )

            col_base = s * 3

            # Original
            ax0 = fig.add_subplot(gs[row, col_base])
            ax0.imshow(original_rgb)
            ax0.set_title(f'True: {cls_name}\nPred: {pred_label} '
                          f'({pred_conf:.1f}%)', fontsize=9)
            ax0.axis('off')

            # Activation map
            ax1 = fig.add_subplot(gs[row, col_base + 1])
            ax1.imshow(heatmap_up, cmap='jet', vmin=0, vmax=1)
            ax1.set_title('Activation Map', fontsize=9)
            ax1.axis('off')

            # Overlay on original
            ax2 = fig.add_subplot(gs[row, col_base + 2])
            ax2.imshow(superimposed)
            ax2.set_title('Overlay on Original', fontsize=9)
            ax2.axis('off')

    fig.suptitle(f'{model_name.upper()} — Grad-CAM Explanations',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_path = os.path.join(output_dir, f'{model_name}_gradcam.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Saved: {save_path}")


def run_gradcam_all(model_paths, test_dir=TEST_DIR,
                    output_dir=OUTPUT_DIR, samples_per_class=2):
    """Run Grad-CAM for every trained model."""
    os.makedirs(output_dir, exist_ok=True)
    for name, path in model_paths.items():
        if not os.path.exists(path):
            print(f"  ❌ Not found: {path}"); continue
        model = tf.keras.models.load_model(path)
        run_gradcam_for_model(model, name, test_dir, output_dir,
                              samples_per_class)


print("✅ Grad-CAM functions ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 9 — LIME (Paper-faithful with Quickshift)           ║
# ╚═══════════════════════════════════════════════��═══════════════╝
# Flowchart steps:
#   1. Superpixel generation  →  Quickshift  →  {S1, S2, …, Sn}
#   2. Perturbation           →  {z'1, …, z'n} + π(x,z)  (Eq 15)
#   3. Prediction + Local linear model                    (Eq 16–17)
#   4. Rank superpixels → colour-coded explanation
#      Green = positive influence, Red = negative influence
# ═════════════════════════════════════════════════════════════════

def _make_predict_fn(model):
    """Wraps model.predict for LIME (expects uint8 input)."""
    def predict_fn(images):
        images = np.array(images, dtype=np.float32) / 255.0
        return model.predict(images, verbose=0)
    return predict_fn


def show_quickshift_segments(img_uint8, ax=None):
    """
    STEP 1: Superpixel generation using Quickshift.
    Segments based on pixel intensity and spatial proximity.
    (Paper Figure 3)
    """
    segments = quickshift(img_uint8, kernel_size=4,
                          max_dist=200, ratio=0.2)
    n_segments = len(np.unique(segments))
    if ax is not None:
        ax.imshow(mark_boundaries(img_uint8 / 255.0, segments,
                                  color=(1, 1, 0), mode='thick'))
        ax.set_title(f'Quickshift\n({n_segments} superpixels)', fontsize=9)
        ax.axis('off')
    return segments


def run_lime_for_model(model, model_name, test_dir=TEST_DIR,
                       output_dir=OUTPUT_DIR, samples_per_class=2,
                       num_samples=1000, num_features=5):
    """
    Full LIME pipeline per the paper.
    Generates 5-column grid per sample:
      Col 0 : Original image + prediction
      Col 1 : Quickshift superpixels             (Step 1)
      Col 2 : Top-5 positive superpixels (green) (Step 4)
      Col 3 : Positive (green) + Negative (red)  (Step 4)
      Col 4 : Superpixel weight heatmap (w)       (learned weights)
    """
    print(f"\n  🍋 LIME for {model_name.upper()} …")

    predict_fn = _make_predict_fn(model)

    # Force Quickshift as segmenter (paper requirement)
    segmenter = SegmentationAlgorithm(
        'quickshift', kernel_size=4, max_dist=200, ratio=0.2,
    )
    explainer = lime_image.LimeImageExplainer(random_state=42)

    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 5

    fig = plt.figure(figsize=(4.5 * n_cols, 5 * n_rows))
    gs  = gridspec.GridSpec(n_rows, n_cols, wspace=0.08, hspace=0.35)

    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = sorted(glob.glob(os.path.join(cls_dir, '*')))[:samples_per_class]

        for s, img_path in enumerate(img_files):
            # Load image as uint8 — LIME expects this
            img = keras_image.load_img(img_path, target_size=IMG_SIZE)
            img_uint8 = keras_image.img_to_array(img).astype(np.uint8)

            # Model prediction for title
            img_norm = img_uint8.astype(np.float32) / 255.0
            preds = model.predict(np.expand_dims(img_norm, 0), verbose=0)
            pred_idx   = int(np.argmax(preds[0]))
            pred_conf  = float(preds[0][pred_idx]) * 100
            pred_label = CLASS_NAMES[pred_idx]

            # ── STEPS 1–3: Quickshift → Perturb → Local model ──
            # Internally:
            #   Step 1: quickshift → {S1, …, Sn}
            #   Step 2: perturb superpixels, weight by
            #           π(x,z) = exp(-dist(x,z)²/σ²)    (Eq 15)
            #   Step 3: fit local linear model
            #           f(z'i) = wᵀ z'i + b              (Eq 16)
            #           minimise Σ π·[f(x)−f(z'i)]²      (Eq 17)
            explanation = explainer.explain_instance(
                img_uint8,
                predict_fn,
                top_labels=NUM_CLASSES,
                hide_color=0,
                num_samples=num_samples,
                segmentation_fn=segmenter,
                random_seed=42,
            )

            col_base = s * 5

            # ── Col 0: Original ──────────────────────────────
            ax0 = fig.add_subplot(gs[row, col_base])
            ax0.imshow(img_uint8)
            ax0.set_title(f'True: {cls_name}\nPred: {pred_label} '
                          f'({pred_conf:.1f}%)', fontsize=9)
            ax0.axis('off')

            # ── Col 1: Quickshift superpixels (Step 1) ───────
            ax1 = fig.add_subplot(gs[row, col_base + 1])
            show_quickshift_segments(img_uint8, ax=ax1)

            # ── STEP 4: Rank superpixels by |w| ─────────────

            # ── Col 2: Top-5 positive (GREEN) ────────────────
            temp_pos, mask_pos = explanation.get_image_and_mask(
                label=pred_idx,
                positive_only=True,
                num_features=num_features,
                hide_rest=False,
            )
            ax2 = fig.add_subplot(gs[row, col_base + 2])
            ax2.imshow(mark_boundaries(
                temp_pos / 255.0, mask_pos,
                color=(0, 1, 0), mode='thick',
            ))
            ax2.set_title(f'Positive\n(top {num_features})', fontsize=9)
            ax2.axis('off')

            # ── Col 3: Positive (green) + Negative (red) ────
            temp_pn, mask_pn = explanation.get_image_and_mask(
                label=pred_idx,
                positive_only=False,
                num_features=num_features,
                hide_rest=False,
            )
            _, mask_pos_only = explanation.get_image_and_mask(
                label=pred_idx,
                positive_only=True,
                num_features=num_features,
                hide_rest=False,
            )
            neg_mask = (mask_pn == 1) & (mask_pos_only == 0)
            display = temp_pn / 255.0
            display = mark_boundaries(display, mask_pos_only,
                                      color=(0, 1, 0), mode='thick')
            display = mark_boundaries(display, neg_mask.astype(int),
                                      color=(1, 0, 0), mode='thick')
            ax3 = fig.add_subplot(gs[row, col_base + 3])
            ax3.imshow(display)
            ax3.set_title('Pos (green)\n+ Neg (red)', fontsize=9)
            ax3.axis('off')

            # ── Col 4: Weight heatmap (learned w) ────────────
            dict_heatmap = dict(explanation.local_exp[pred_idx])
            weight_map = np.vectorize(
                lambda seg_id: dict_heatmap.get(seg_id, 0.0)
            )(explanation.segments)

            ax4 = fig.add_subplot(gs[row, col_base + 4])
            vmax = np.max(np.abs(weight_map)) or 1.0
            im = ax4.imshow(weight_map, cmap='RdYlGn',
                            vmin=-vmax, vmax=vmax)
            ax4.set_title('Weight Heatmap\n(learned w)', fontsize=9)
            ax4.axis('off')
            plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

    fig.suptitle(
        f'{model_name.upper()} — LIME Explanations (Quickshift Superpixels)',
        fontsize=16, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    save_path = os.path.join(output_dir, f'{model_name}_lime.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Saved: {save_path}")


def plot_superpixel_weights(model, img_path, model_name,
                            output_dir=OUTPUT_DIR,
                            num_samples=1000, top_n=15):
    """
    Paper Figure 4 — bar chart of superpixel weights
    from the local linear model.
    """
    predict_fn = _make_predict_fn(model)
    segmenter  = SegmentationAlgorithm(
        'quickshift', kernel_size=4, max_dist=200, ratio=0.2,
    )
    explainer = lime_image.LimeImageExplainer(random_state=42)

    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    img_uint8 = keras_image.img_to_array(img).astype(np.uint8)

    img_norm = img_uint8.astype(np.float32) / 255.0
    preds = model.predict(np.expand_dims(img_norm, 0), verbose=0)
    pred_idx = int(np.argmax(preds[0]))

    explanation = explainer.explain_instance(
        img_uint8, predict_fn,
        top_labels=NUM_CLASSES, hide_color=0,
        num_samples=num_samples, segmentation_fn=segmenter,
        random_seed=42,
    )

    weights = explanation.local_exp[pred_idx]
    weights_sorted = sorted(weights, key=lambda x: abs(x[1]),
                            reverse=True)[:top_n]
    seg_ids = [f'S{w[0]}' for w in weights_sorted]
    values  = [w[1] for w in weights_sorted]
    colours = ['green' if v > 0 else 'red' for v in values]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(seg_ids[::-1], values[::-1], color=colours[::-1],
            edgecolor='white')
    ax.set_xlabel('Weight (w)', fontsize=11)
    ax.set_ylabel('Superpixel', fontsize=11)
    ax.set_title(f'{model_name.upper()} — Superpixel Weights '
                 f'(Pred: {CLASS_NAMES[pred_idx]})', fontsize=13)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    save_path = os.path.join(output_dir,
                             f'{model_name}_superpixel_weights.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Superpixel weight chart saved: {save_path}")


def run_lime_all(model_paths, test_dir=TEST_DIR,
                 output_dir=OUTPUT_DIR, samples_per_class=2,
                 num_samples=1000, num_features=5):
    """Run LIME for every trained model."""
    os.makedirs(output_dir, exist_ok=True)
    for name, path in model_paths.items():
        if not os.path.exists(path):
            print(f"  ❌ Not found: {path}"); continue
        model = tf.keras.models.load_model(path)
        run_lime_for_model(model, name, test_dir, output_dir,
                           samples_per_class, num_samples, num_features)


print("✅ LIME functions ready.")


# ╔═══════════════════════════════════════════════════════════════╗
# ║  SECTION 10 — RUN EVERYTHING                                 ║
# ╚═══════════════════════════════════════════════════════════════╝

print("\n" + "=" * 60)
print("  🧠 BRAIN TUMOR CLASSIFICATION — FULL PIPELINE")
print("=" * 60)

# ── PHASE 1: TRAIN ALL MODELS ────────────────────────────────
print("\n" + "=" * 60)
print("  📚 PHASE 1 — TRAINING")
print("=" * 60)
trained = train_all(MODEL_NAMES, TRAIN_DIR, OUTPUT_DIR)

# ── PHASE 2: EVALUATE ALL MODELS ─────────────────────────────
print("\n" + "=" * 60)
print("  📊 PHASE 2 — EVALUATION")
print("=" * 60)
model_paths = {}
for name in MODEL_NAMES:
    p = os.path.join(OUTPUT_DIR, f"{name}_final.keras")
    if os.path.exists(p):
        model_paths[name] = p

if model_paths:
    all_metrics = evaluate_all(model_paths, TEST_DIR, OUTPUT_DIR)
else:
    print("  ❌ No trained models found!")

# ── PHASE 3: GRAD-CAM ────────────────────────────────────────
print("\n" + "=" * 60)
print("  🔥 PHASE 3 — GRAD-CAM")
print("  Steps: Gradient → GAP → Weight+ReLU → Upsample+Overlay")
print("=" * 60)
if model_paths:
    run_gradcam_all(model_paths, TEST_DIR, OUTPUT_DIR, samples_per_class=2)

# ── PHASE 4: LIME ────────────────────────────────────────────
print("\n" + "=" * 60)
print("  🍋 PHASE 4 — LIME")
print("  Steps: Quickshift → Perturb → Local Model → Rank → Colour")
print("=" * 60)
if model_paths:
    run_lime_all(model_paths, TEST_DIR, OUTPUT_DIR,
                 samples_per_class=2, num_samples=1000, num_features=5)

    # Superpixel weight bar charts (Paper Figure 4)
    print("\n  📊 Generating superpixel weight charts (Paper Fig 4)…")
    for name, path in model_paths.items():
        model = tf.keras.models.load_model(path)
        sample_dir = os.path.join(TEST_DIR, CLASS_NAMES[0])
        sample_img = sorted(glob.glob(os.path.join(sample_dir, '*')))[0]
        plot_superpixel_weights(model, sample_img, name,
                                OUTPUT_DIR, num_samples=1000, top_n=15)

# ── DONE ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  🎉 ALL DONE!")
print("=" * 60)
print(f"\n  All outputs saved to: {OUTPUT_DIR}")
print("  Files generated:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"    📄 {f}  ({size_mb:.1f} MB)")

✅ Dependencies installed.


2026-03-11 15:44:35.191981: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773243875.413162      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773243875.481696      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773243875.972822      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773243875.972859      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773243875.972861      55 computation_placer.cc:177] computation placer alr

✅ All imports loaded.
   TensorFlow version : 2.19.0
   GPU available      : True

✅ Configuration loaded.
   Classes      : ['glioma', 'meningioma', 'notumor', 'pituitary']
   Image size   : (224, 224)
   Batch size   : 32
   Epochs       : 30
   Train dir    : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
   Test dir     : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
   Output dir   : /kaggle/working/outputs

✅ Data preprocessing helpers ready.
✅ Model builder ready.
✅ Training functions ready.
✅ Evaluation functions ready.
✅ Grad-CAM functions ready.
✅ LIME functions ready.

  🧠 BRAIN TUMOR CLASSIFICATION — FULL PIPELINE

  📚 PHASE 1 — TRAINING

  🚀 TRAINING: VGG16
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
   Training samples   : 4480
   Validation samples : 1120


I0000 00:00:1773243903.231592      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773243903.237540      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

   📐 VGG16
      Total params     : 14,848,068
      Trainable params : 132,868
Epoch 1/30


...

  💾 Superpixel weight chart saved: /kaggle/working/outputs/vgg16_superpixel_weights.png


  0%|          | 0/1000 [00:00<?, ?it/s]

  💾 Superpixel weight chart saved: /kaggle/working/outputs/resnet50_superpixel_weights.png


  0%|          | 0/1000 [00:00<?, ?it/s]

  💾 Superpixel weight chart saved: /kaggle/working/outputs/efficientnetb0_superpixel_weights.png

  🎉 ALL DONE!

  All outputs saved to: /kaggle/working/outputs
  Files generated:
    📄 efficientnetb0_best.keras  (20.0 MB)
    📄 efficientnetb0_confusion_matrix.png  (0.1 MB)
    📄 efficientnetb0_final.keras  (20.0 MB)
    📄 efficientnetb0_gradcam.png  (3.3 MB)
    📄 efficientnetb0_lime.png  (3.2 MB)
    📄 efficientnetb0_superpixel_weights.png  (0.0 MB)
    📄 efficientnetb0_training_history.png  (0.1 MB)
    📄 model_comparison.png  (0.1 MB)
    📄 resnet50_best.keras  (96.6 MB)
    📄 resnet50_confusion_matrix.png  (0.1 MB)
    📄 resnet50_final.keras  (96.6 MB)
    📄 resnet50_gradcam.png  (3.3 MB)
    📄 resnet50_lime.png  (3.2 MB)
    📄 resnet50_superpixel_weights.png  (0.0 MB)
    📄 resnet50_training_history.png  (0.1 MB)
    📄 vgg16_best.keras  (57.7 MB)
    📄 vgg16_confusion_matrix.png  (0.1 MB)
    📄 vgg16_final.keras  (57.7 MB)
    📄 vgg16_gradcam.png  (3.7 MB)
    📄 vgg16_lime.png  (

In [2]:
# =============================================================
# 🧠 BRAIN TUMOR CLASSIFICATION — COMPLETE FIXED KAGGLE NOTEBOOK
# Targets: VGG16≥96%, ResNet50≥97%, EfficientNetB0≥97.7%
# Fixes: Correct preprocessing per model, 2-phase training,
#        fine-tuning, label smoothing, proper LR schedule
# =============================================================


# ╔══════════════════════════════════════════╗
# ║  SECTION 1 — INSTALL                    ║
# ╚══════════════════════════════════════════╝
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime', '-q'])
print("✅ Dependencies installed.")


# ╔══════════════════════════════════════════╗
# ║  SECTION 2 — IMPORTS                    ║
# ╚══════════════════════════════════════════╝
import os, glob, warnings, random, math
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing      import image as keras_image
from tensorflow.keras.applications       import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.applications.vgg16    import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.models  import Model
from tensorflow.keras.layers  import (
    Dense, Dropout, GlobalAveragePooling2D,
    BatchNormalization, Activation, Input,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks  import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)
from tensorflow.keras import regularizers

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)
from scipy.stats import kruskal

from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries, quickshift

warnings.filterwarnings('ignore')

# ✅ Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ All imports loaded.")
print(f"   TensorFlow : {tf.__version__}")
print(f"   GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")


# ╔══════════════════════════════════════════╗
# ║  SECTION 3 — CONFIGURATION              ║
# ╚══════════════════════════════════════════╝
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES  = len(CLASS_NAMES)
MODEL_NAMES  = ['vgg16', 'resnet50', 'efficientnetb0']

# ✅ FIX BUG 2: Each model has its ideal input size
MODEL_IMG_SIZES = {
    'vgg16':          (224, 224),
    'resnet50':       (224, 224),
    'efficientnetb0': (224, 224),   # min=224, can use 260 for better accuracy
}

BATCH_SIZE    = 32
PHASE1_EPOCHS = 20    # head only (frozen base)
PHASE2_EPOCHS = 30    # fine-tuning (unfrozen top layers)

# ✅ FIX BUG 6: Separate LRs for each phase
PHASE1_LR = 1e-3      # Higher LR for head training
PHASE2_LR = 1e-5      # Very low LR for fine-tuning

# ✅ FIX BUG 8: Monitor val_accuracy not val_loss
EARLY_STOP_MONITOR = 'val_accuracy'
EARLY_STOP_PATIENCE_P1 = 10
EARLY_STOP_PATIENCE_P2 = 12

LABEL_SMOOTHING = 0.1
L2_REG          = 1e-4
DROPOUT_RATE    = 0.5

TRAIN_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAPER_TARGETS = {
    'vgg16':          {'overall': 96.08},
    'resnet50':       {'overall': 97.06},
    'efficientnetb0': {'overall': 97.71},
}

# ✅ How many top layers to unfreeze per model in Phase 2
UNFREEZE_LAYERS = {
    'vgg16':          'block5_conv1',   # unfreeze block5 (last conv block)
    'resnet50':       'conv5_block1_1_conv',  # unfreeze last residual block
    'efficientnetb0': 20,               # unfreeze last 20 layers (int = count)
}

print("✅ Configuration loaded.")
print(f"   Train dir  : {TRAIN_DIR}")
print(f"   Test dir   : {TEST_DIR}")
print(f"   Output dir : {OUTPUT_DIR}")

# Verify paths
for p in [TRAIN_DIR, TEST_DIR]:
    if os.path.exists(p):
        print(f"   ✅ Found: {p} → {os.listdir(p)}")
    else:
        print(f"   ❌ NOT FOUND: {p}  ← Update this path!")


# ╔══════════════════════════════════════════╗
# ║  SECTION 4 — DATA PREPROCESSING         ║
# ╚══════════════════════════════════════════╝

# ✅ FIX BUG 1 & 5: Each model gets its OWN correct preprocessor
def get_preprocess_fn(model_name):
    """
    Returns the correct preprocessing function per model:
    - VGG16      → subtracts ImageNet BGR mean [103.9, 116.8, 123.7]
    - ResNet50   → subtracts ImageNet BGR mean (same as VGG16)
    - EfficientNetB0 → NO preprocessing (internal normalization 0-255)
    """
    if model_name == 'vgg16':
        return vgg16_preprocess       # expects BGR, subtracts mean
    elif model_name == 'resnet50':
        return resnet50_preprocess    # expects BGR, subtracts mean
    elif model_name == 'efficientnetb0':
        return None                   # ✅ NO rescale — handles internally
    raise ValueError(f"Unknown model: {model_name}")


def get_generators(model_name, train_dir=TRAIN_DIR, test_dir=TEST_DIR):
    """
    Returns (train_gen, val_gen, test_gen) with model-specific preprocessing.
    80% train / 20% val split. No data leakage.
    """
    img_size      = MODEL_IMG_SIZES[model_name]
    preprocess_fn = get_preprocess_fn(model_name)

    # ✅ FIX BUG 1: Use preprocessing_function, NOT rescale
    if preprocess_fn is not None:
        # VGG16 / ResNet50 — use their specific preprocessors
        train_datagen = ImageDataGenerator(
            preprocessing_function = preprocess_fn,
            rotation_range         = 20,
            width_shift_range      = 0.15,
            height_shift_range     = 0.15,
            shear_range            = 0.1,
            zoom_range             = 0.15,
            horizontal_flip        = True,
            vertical_flip          = True,
            brightness_range       = [0.8, 1.2],
            fill_mode              = 'nearest',
            validation_split       = 0.2,
        )
        test_datagen = ImageDataGenerator(
            preprocessing_function = preprocess_fn,  # same for test!
        )
    else:
        # EfficientNetB0 — NO preprocessing at all
        train_datagen = ImageDataGenerator(
            rotation_range     = 20,
            width_shift_range  = 0.15,
            height_shift_range = 0.15,
            shear_range        = 0.1,
            zoom_range         = 0.15,
            horizontal_flip    = True,
            vertical_flip      = True,
            brightness_range   = [0.8, 1.2],
            fill_mode          = 'nearest',
            validation_split   = 0.2,
        )
        test_datagen = ImageDataGenerator()  # NO rescale for EfficientNet

    common = dict(
        target_size = img_size,
        batch_size  = BATCH_SIZE,
        class_mode  = 'categorical',
        classes     = CLASS_NAMES,
        seed        = SEED,
    )

    train_gen = train_datagen.flow_from_directory(
        train_dir, subset='training',   shuffle=True,  **common)
    val_gen   = train_datagen.flow_from_directory(
        train_dir, subset='validation', shuffle=False, **common)
    test_gen  = test_datagen.flow_from_directory(
        test_dir, shuffle=False, **common)

    print(f"\n   [{model_name.upper()}] Data generators ready:")
    print(f"     Train : {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")
    print(f"     Img size: {img_size} | Classes: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen


print("✅ Data preprocessing ready.")


# ╔══════════════════════════════════════════╗
# ║  SECTION 5 — MODEL BUILDING             ║
# ╚══════════════════════════════════════════╝

def build_model(model_name):
    """
    Phase 1 model: frozen base + improved classification head.
    Head: GAP → Dense(512)→BN→ReLU→Drop(0.5) → Dense(256)→BN→ReLU→Drop(0.3) → softmax
    """
    img_size     = MODEL_IMG_SIZES[model_name]
    input_shape  = (*img_size, 3)
    base_kwargs  = dict(include_top=False, weights='imagenet',
                        input_shape=input_shape)

    if model_name == 'vgg16':
        base = VGG16(**base_kwargs)
    elif model_name == 'resnet50':
        base = ResNet50(**base_kwargs)
    elif model_name == 'efficientnetb0':
        base = EfficientNetB0(**base_kwargs)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # ✅ FIX BUG 3 & 4: Freeze ALL layers for Phase 1
    base.trainable = False

    # ✅ FIX BUG 7: Improved head with L2 regularization
    inputs  = Input(shape=input_shape)

    # For EfficientNetB0, pass training=False to keep BN frozen
    if model_name == 'efficientnetb0':
        x = base(inputs, training=False)
    else:
        x = base(inputs)

    x = GlobalAveragePooling2D(name='gap')(x)

    x = Dense(512, kernel_regularizer=regularizers.l2(L2_REG),
              name='fc_512')(x)
    x = BatchNormalization(name='bn_1')(x)
    x = Activation('relu', name='relu_1')(x)
    x = Dropout(DROPOUT_RATE, name='drop_1')(x)

    x = Dense(256, kernel_regularizer=regularizers.l2(L2_REG),
              name='fc_256')(x)
    x = BatchNormalization(name='bn_2')(x)
    x = Activation('relu', name='relu_2')(x)
    x = Dropout(0.3, name='drop_2')(x)

    outputs = Dense(NUM_CLASSES, activation='softmax',
                    name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs,
                  name=f'{model_name}_BrainTumor')

    # ✅ FIX BUG 6 & 9: Higher LR for Phase 1 + label smoothing
    model.compile(
        optimizer = Adam(learning_rate=PHASE1_LR),
        loss      = tf.keras.losses.CategoricalCrossentropy(
                        label_smoothing=LABEL_SMOOTHING),
        metrics   = ['accuracy'],
    )

    trainable = sum(tf.keras.backend.count_params(w)
                    for w in model.trainable_weights)
    print(f"\n   📐 {model_name.upper()} Phase 1 model built.")
    print(f"      Input shape      : {input_shape}")
    print(f"      Trainable params : {trainable:,}  (head only)")
    return model


def unfreeze_model_for_phase2(model, model_name):
    """
    Phase 2: unfreeze top layers of base for fine-tuning.
    VGG16/ResNet50: unfreeze from specific layer onwards.
    EfficientNetB0: unfreeze last N layers.
    BatchNorm layers always stay FROZEN.
    """
    unfreeze_cfg = UNFREEZE_LAYERS[model_name]

    # Get the base sub-model
    base = None
    for layer in model.layers:
        if hasattr(layer, 'layers'):   # is a sub-model
            base = layer
            break

    if base is None:
        print(f"  ⚠️  Could not find base sub-model for {model_name}")
        return model

    if model_name in ['vgg16', 'resnet50']:
        # ✅ Unfreeze from specific named layer onwards
        base.trainable = True
        trainable_flag = False
        for layer in base.layers:
            if layer.name == unfreeze_cfg:
                trainable_flag = True
            if isinstance(layer, BatchNormalization):
                layer.trainable = False   # ALWAYS keep BN frozen
            else:
                layer.trainable = trainable_flag

    elif model_name == 'efficientnetb0':
        # ✅ Unfreeze last N layers
        base.trainable = True
        total = len(base.layers)
        for i, layer in enumerate(base.layers):
            if isinstance(layer, BatchNormalization):
                layer.trainable = False   # ALWAYS keep BN frozen
            else:
                layer.trainable = (i >= total - unfreeze_cfg)

    # ✅ FIX BUG 6: Very low LR for fine-tuning
    model.compile(
        optimizer = Adam(learning_rate=PHASE2_LR),
        loss      = tf.keras.losses.CategoricalCrossentropy(
                        label_smoothing=LABEL_SMOOTHING),
        metrics   = ['accuracy'],
    )

    trainable = sum(tf.keras.backend.count_params(w)
                    for w in model.trainable_weights)
    print(f"   ✅ {model_name.upper()} Phase 2: top layers unfrozen.")
    print(f"      Trainable params now : {trainable:,}")
    return model


print("✅ Model builder ready.")


# ╔══════════════════════════════════════════╗
# ║  SECTION 6 — TRAINING (2-PHASE)         ║
# ╚══════════════════════════════════════════╝

def get_callbacks(model_name, phase, output_dir=OUTPUT_DIR):
    """Phase-aware callbacks."""
    ckpt = os.path.join(output_dir,
                        f'{model_name}_phase{phase}_best.keras')
    patience = (EARLY_STOP_PATIENCE_P1 if phase == 1
                else EARLY_STOP_PATIENCE_P2)
    return [
        ModelCheckpoint(
            ckpt, monitor=EARLY_STOP_MONITOR,
            save_best_only=True, mode='max', verbose=1),
        EarlyStopping(
            monitor=EARLY_STOP_MONITOR, patience=patience,
            restore_best_weights=True, mode='max', verbose=1),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.3,
            patience=3, min_lr=1e-8, verbose=1),
    ]


def plot_history(h1, h2, model_name, output_dir=OUTPUT_DIR):
    """Combined Phase 1 + Phase 2 training curves."""
    acc      = h1.history['accuracy']     + h2.history['accuracy']
    val_acc  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss     = h1.history['loss']         + h2.history['loss']
    val_loss = h1.history['val_loss']     + h2.history['val_loss']
    epochs   = range(1, len(acc) + 1)
    split    = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f'{model_name.upper()} — Training History',
                 fontsize=14, fontweight='bold')

    axes[0].plot(epochs, acc,     'b-', label='Train Acc', linewidth=2)
    axes[0].plot(epochs, val_acc, 'r-', label='Val Acc',   linewidth=2)
    axes[0].axvline(split, color='green', linestyle='--',
                    linewidth=1.5, label='Fine-tune start')
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, loss,     'b-', label='Train Loss', linewidth=2)
    axes[1].plot(epochs, val_loss, 'r-', label='Val Loss',   linewidth=2)
    axes[1].axvline(split, color='green', linestyle='--',
                    linewidth=1.5, label='Fine-tune start')
    axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, f'{model_name}_training_history.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Training curves saved: {path}")


def train_model(model_name, train_dir=TRAIN_DIR, output_dir=OUTPUT_DIR):
    """Full 2-phase training: head-only → fine-tune."""
    print(f"\n{'='*60}")
    print(f"  🚀 TRAINING: {model_name.upper()}")
    print(f"{'='*60}")

    train_gen, val_gen, _ = get_generators(model_name, train_dir)
    model = build_model(model_name)
    model.summary(line_length=90)

    # ── PHASE 1: Train head only (base frozen) ─────────────
    print(f"\n  📌 Phase 1: head only, "
          f"lr={PHASE1_LR}, max {PHASE1_EPOCHS} epochs …")
    cb_p1      = get_callbacks(model_name, phase=1, output_dir=output_dir)
    history_p1 = model.fit(
        train_gen, epochs=PHASE1_EPOCHS,
        validation_data=val_gen,
        callbacks=cb_p1, verbose=1,
    )
    p1_best = max(history_p1.history['val_accuracy'])
    print(f"\n  ✅ Phase 1 best val_accuracy: {p1_best*100:.2f}%")

    # ── PHASE 2: Fine-tune top layers ──────────────────────
    print(f"\n  📌 Phase 2: fine-tuning, "
          f"lr={PHASE2_LR}, max {PHASE2_EPOCHS} epochs …")
    model     = unfreeze_model_for_phase2(model, model_name)
    cb_p2     = get_callbacks(model_name, phase=2, output_dir=output_dir)
    history_p2 = model.fit(
        train_gen, epochs=PHASE2_EPOCHS,
        validation_data=val_gen,
        callbacks=cb_p2, verbose=1,
    )
    p2_best = max(history_p2.history['val_accuracy'])
    print(f"\n  ✅ Phase 2 best val_accuracy: {p2_best*100:.2f}%")

    # Save final
    final_path = os.path.join(output_dir, f'{model_name}_final.keras')
    model.save(final_path)
    print(f"  💾 Model saved: {final_path}")

    plot_history(history_p1, history_p2, model_name, output_dir)

    overall_best = max(p1_best, p2_best)
    target       = PAPER_TARGETS[model_name]['overall']
    total_ep     = (len(history_p1.history['accuracy']) +
                    len(history_p2.history['accuracy']))
    print(f"\n  📊 {model_name.upper()} Summary:")
    print(f"     Total epochs run  : {total_ep}")
    print(f"     Best val accuracy : {overall_best*100:.2f}%")
    print(f"     Paper target      : {target}%")
    gap = overall_best*100 - target
    print(f"     Gap vs paper      : {gap:+.2f}%")

    return model, history_p1, history_p2


def train_all(model_names=MODEL_NAMES, train_dir=TRAIN_DIR,
              output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    trained = {}
    for name in model_names:
        model, h1, h2 = train_model(name, train_dir, output_dir)
        trained[name] = {'model': model, 'h1': h1, 'h2': h2}
    print(f"\n  🎉 All {len(trained)} models trained!")
    return trained


print("✅ Training functions ready.")


# ╔══════════════════════════════════════════╗
# ║  SECTION 7 — EVALUATION                 ║
# ╚══════════════════════════════════════════╝

def predict(model, test_gen):
    test_gen.reset()
    steps  = int(np.ceil(test_gen.samples / BATCH_SIZE))
    probs  = model.predict(test_gen, steps=steps, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_gen.classes[:len(y_pred)]
    return y_true, y_pred


def per_class_accuracy(y_true, y_pred):
    return {
        name: accuracy_score(y_true[y_true == i], y_pred[y_true == i])
        for i, name in enumerate(CLASS_NAMES)
        if (y_true == i).sum() > 0
    }


def plot_confusion_matrix(y_true, y_pred, model_name,
                           output_dir=OUTPUT_DIR):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{model_name.upper()} — Confusion Matrix',
                 fontsize=13)
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    path = os.path.join(output_dir,
                        f'{model_name}_confusion_matrix.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾 Confusion matrix: {path}")


def evaluate_model(model, model_name, test_dir=TEST_DIR,
                   output_dir=OUTPUT_DIR):
    print(f"\n  🔍 Evaluating: {model_name.upper()} …")
    # ✅ FIX: each model gets its OWN test generator with correct preprocessing
    _, _, test_gen = get_generators(model_name,
                                    train_dir=TRAIN_DIR,
                                    test_dir=test_dir)
    y_true, y_pred = predict(model, test_gen)

    overall     = accuracy_score(y_true, y_pred)
    cls_accs    = per_class_accuracy(y_true, y_pred)
    macro_prec  = precision_score(y_true, y_pred,
                                  average='macro', zero_division=0)
    macro_rec   = recall_score(y_true, y_pred,
                               average='macro', zero_division=0)
    macro_f1    = f1_score(y_true, y_pred,
                           average='macro', zero_division=0)
    target      = PAPER_TARGETS.get(model_name, {}).get('overall', 'N/A')

    print(f"\n  ─── {model_name.upper()} Results ───")
    print(f"  Overall Accuracy : {overall*100:.2f}%  (paper: {target}%)")
    print(f"  Macro Precision  : {macro_prec*100:.2f}%")
    print(f"  Macro Recall     : {macro_rec*100:.2f}%")
    print(f"  Macro F1         : {macro_f1*100:.2f}%")
    print(f"\n  Per-Class Accuracy:")
    for cls, acc in cls_accs.items():
        print(f"      {cls:12s}: {acc*100:.2f}%")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES,
                                zero_division=0))
    plot_confusion_matrix(y_true, y_pred, model_name, output_dir)

    return {
        'model_name': model_name, 'overall_acc': overall,
        'cls_accs': cls_accs,
        'macro_precision': macro_prec, 'macro_recall': macro_rec,
        'macro_f1': macro_f1,
        'y_true': y_true, 'y_pred': y_pred,
    }


def plot_model_comparison(metrics_list, output_dir=OUTPUT_DIR):
    names        = [m['model_name'].upper() for m in metrics_list]
    overall_accs = [m['overall_acc'] * 100   for m in metrics_list]
    targets      = [PAPER_TARGETS[m['model_name']]['overall']
                    for m in metrics_list]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Model Comparison — Brain Tumor Classification',
                 fontsize=14, fontweight='bold')

    colours = ['#4C72B0', '#DD8452', '#55A868']
    bars    = axes[0].bar(names, overall_accs, color=colours,
                          edgecolor='white', width=0.5)
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=11)
    # Paper target lines
    for i, t in enumerate(targets):
        axes[0].hlines(t, i - 0.3, i + 0.3,
                       colors='red', linestyles='--', linewidth=1.5)
    axes[0].set_title('Overall Accuracy (red dashed = paper target)')
    axes[0].set_ylim(80, 105)
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].legend(['Paper target'], loc='lower right')

    x = np.arange(len(CLASS_NAMES))
    w = 0.22
    for i, m in enumerate(metrics_list):
        accs = [m['cls_accs'].get(c, 0) * 100 for c in CLASS_NAMES]
        off  = (i - len(metrics_list)/2 + 0.5) * w
        b    = axes[1].bar(x + off, accs, w,
                           label=names[i], color=colours[i])
        axes[1].bar_label(b, fmt='%.1f', padding=2, fontsize=7)
    axes[1].set_title('Per-Class Accuracy (%)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c.capitalize() for c in CLASS_NAMES])
    axes[1].set_ylim(70, 108)
    axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, 'model_comparison.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n  💾 Model comparison: {path}")


def kruskal_wallis_test(metrics_list):
    groups = [list(m['cls_accs'].values()) for m in metrics_list]
    if len(groups) < 2:
        return {}
    h, p   = kruskal(*groups)
    sig    = '(significant p<0.05)' if p < 0.05 else '(not significant)'
    print(f"\n  📊 Kruskal-Wallis: H={h:.4f}, p={p:.4f}  {sig}")
    print(f"     Paper reference : H=7.88, p=0.019")
    return {'statistic': h, 'pvalue': p}


def evaluate_all(model_paths, test_dir=TEST_DIR,
                 output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    all_metrics = []
    for model_name, path in model_paths.items():
        if not os.path.exists(path):
            print(f"  ❌ Not found: {path}"); continue
        print(f"\n  📂 Loading: {path}")
        model   = tf.keras.models.load_model(path)
        metrics = evaluate_model(model, model_name,
                                 test_dir, output_dir)
        all_metrics.append(metrics)
    if len(all_metrics) >= 2:
        plot_model_comparison(all_metrics, output_dir)
        kruskal_wallis_test(all_metrics)
    return all_metrics


print("✅ Evaluation functions ready.")


✅ Dependencies installed.
✅ All imports loaded.
   TensorFlow : 2.19.0
   GPU        : True
✅ Configuration loaded.
   Train dir  : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
   Test dir   : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
   Output dir : /kaggle/working/outputs
   ✅ Found: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training → ['pituitary', 'notumor', 'meningioma', 'glioma']
   ✅ Found: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing → ['pituitary', 'notumor', 'meningioma', 'glioma']
✅ Data preprocessing ready.
✅ Model builder ready.
✅ Training functions ready.
✅ Evaluation functions ready.
✅ Grad-CAM functions ready.
✅ LIME functions ready.

  🧠 BRAIN TUMOR — FULL PIPELINE

  📚 PHASE 1 — TRAINING (2-phase per model)

  🚀 TRAINING: VGG16
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

   [VGG16]

Model: "vgg16_BrainTumor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)            │ (None, 224, 224, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ vgg16 (Functional)                    │ (None, 7, 7, 512)            │      14,714,688 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ gap (GlobalAveragePooling2D)          │ (None, 512)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ fc_512 (Dense)                        │ (None, 512)                  │         262,656 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ bn_1 (BatchNormalization)             │ (None, 512)                  │           2,048 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu_1 (Activation)                   │ (None, 512)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ drop_1 (Dropout)                      │ (None, 512)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ fc_256 (Dense)                        │ (None, 256)                  │         131,328 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ bn_2 (BatchNormalization)             │ (None, 256)                  │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ relu_2 (Activation)                   │ (None, 256)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ drop_2 (Dropout)                      │ (None, 256)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ predictions (Dense)                   │ (None, 4)                    │           1,028 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 15,112,772 (57.65 MB)

 Trainable params: 396,548 (1.51 MB)

...

 Total params: 4,840,871 (18.47 MB)

 Trainable params: 789,764 (3.01 MB)

 Non-trainable params: 4,051,107 (15.45 MB)


  📌 Phase 1: head only, lr=0.001, max 20 epochs …
Epoch 1/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.6806 - loss: 1.1115
Epoch 1: val_accuracy improved from -inf to 0.80804, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 102s 584ms/step - accuracy: 0.6810 - loss: 1.1107 - val_accuracy: 0.8080 - val_loss: 0.8237 - learning_rate: 0.0010
Epoch 2/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.8027 - loss: 0.8688
Epoch 2: val_accuracy improved from 0.80804 to 0.87500, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 74s 528ms/step - accuracy: 0.8028 - loss: 0.8686 - val_accuracy: 0.8750 - val_loss: 0.7401 - learning_rate: 0.0010
Epoch 3/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step - accuracy: 0.8292 - loss: 0.8101
Epoch 3: val_accuracy improved from 0.87500 to 0.88125, saving model to /kaggle/working/outputs/efficientnetb0_phase1_best.keras
140/140 ━━━━

ValueError: No Conv2D found.

In [1]:
# =============================================================
# 🧠 VGG16 + ResNet50 + EfficientNetB0 — ALL FIXED
# Target: VGG16≥96%, ResNet50≥97%, EfficientNetB0≥97%
# Key fixes:
#   1. Class weights (glioma penalty)
#   2. Correct preprocessing per model
#   3. More fine-tuning layers
#   4. Stronger augmentation
#   5. 3-phase training for ResNet50 & EfficientNetB0
# =============================================================

import os, glob, warnings, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.applications.vgg16    import preprocess_input as vgg16_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_pre
from tensorflow.keras.preprocessing.image   import ImageDataGenerator
from tensorflow.keras.models  import Model
from tensorflow.keras.layers  import (
    Dense, Dropout, GlobalAveragePooling2D,
    BatchNormalization, Activation, Input,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks  import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)
from tensorflow.keras import regularizers
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
)
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Shared config ────────────────────────────────────────────
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = 4
BATCH_SIZE  = 32
OUTPUT_DIR  = '/kaggle/working/outputs'
TRAIN_DIR   = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Previous results for comparison ─────────────────────────
PREV_RESULTS = {
    'vgg16':    {'overall': 95.44, 'glioma': 86.00,
                 'meningioma': 96.25, 'notumor': 99.75,
                 'pituitary': 99.75},
    'resnet50': {'overall': 93.38, 'glioma': 78.50,
                 'meningioma': 95.50, 'notumor': 100.00,
                 'pituitary': 99.50},
    'efficientnetb0': {'overall': 90.12, 'glioma': 74.75,
                       'meningioma': 87.25, 'notumor': 99.50,
                       'pituitary': 99.00},
}

# ── Per-model config ─────────────────────────────────────────
MODEL_CONFIG = {
    'vgg16': {
        'model_fn'      : VGG16,
        'preprocess_fn' : vgg16_pre,
        'img_size'      : (224, 224),
        # VGG16 is close to target — just needs glioma fix
        # Unfreeze block5 (last conv block = 3 layers)
        'unfreeze_p2'   : 'block5_conv1',  # named layer
        'unfreeze_p3'   : 'block4_conv1',  # go deeper if needed
        'phase1_lr'     : 1e-3,
        'phase2_lr'     : 1e-5,
        'phase3_lr'     : 5e-6,
        'phase1_epochs' : 25,
        'phase2_epochs' : 30,
        'phase3_epochs' : 20,
        'paper_target'  : 96.08,
        'unfreeze_type' : 'named',         # use layer name
    },
    'resnet50': {
        'model_fn'      : ResNet50,
        'preprocess_fn' : resnet50_pre,
        'img_size'      : (224, 224),
        # ResNet50 has big gap — needs more fine-tuning
        # Unfreeze last 2 residual blocks (conv4+conv5)
        'unfreeze_p2'   : 'conv5_block1_1_conv',
        'unfreeze_p3'   : 'conv4_block1_1_conv',
        'phase1_lr'     : 1e-3,
        'phase2_lr'     : 1e-5,
        'phase3_lr'     : 5e-6,
        'phase1_epochs' : 25,
        'phase2_epochs' : 35,
        'phase3_epochs' : 25,
        'paper_target'  : 97.06,
        'unfreeze_type' : 'named',
    },
    'efficientnetb0': {
        'model_fn'      : EfficientNetB0,
        'preprocess_fn' : None,            # NO preprocess
        'img_size'      : (224, 224),
        # EfficientNetB0 has worst gap — needs most fine-tuning
        'unfreeze_p2'   : 50,              # last 50 layers
        'unfreeze_p3'   : 100,             # last 100 layers
        'phase1_lr'     : 1e-3,
        'phase2_lr'     : 1e-5,
        'phase3_lr'     : 5e-6,
        'phase1_epochs' : 25,
        'phase2_epochs' : 35,
        'phase3_epochs' : 25,
        'paper_target'  : 97.71,
        'unfreeze_type' : 'count',         # use layer count
    },
}

print("✅ Config ready")
print(f"   GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")


# ╔══════════════════════════════════════════╗
# ║  HELPER 1 — CLASS WEIGHTS               ║
# ╚══════════════════════════════════════════╝

def compute_class_weights(train_dir):
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(train_dir, cls)
        counts[cls] = len([
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg','.jpeg','.png'))
        ])
    total  = sum(counts.values())
    n_cls  = len(CLASS_NAMES)
    weights = {
        i: total / (n_cls * counts[cls])
        for i, cls in enumerate(CLASS_NAMES)
    }
    print("\n📊 Class weights:")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"   {cls:12s}: {weights[i]:.4f} "
              f"({counts[cls]} images)")
    return weights


# ╔══════════════════════════════════════════╗
# ║  HELPER 2 — DATA GENERATORS             ║
# ╚══════════════════════════════════════════╝

def get_generators(model_name):
    """Each model gets its own correct preprocessing."""
    cfg      = MODEL_CONFIG[model_name]
    img_size = cfg['img_size']
    pre_fn   = cfg['preprocess_fn']

    # ✅ VGG16/ResNet50 → use preprocessing_function
    # ✅ EfficientNetB0 → NO preprocessing (handles internally)
    aug_kwargs = dict(
        rotation_range     = 25,
        width_shift_range  = 0.2,
        height_shift_range = 0.2,
        shear_range        = 0.15,
        zoom_range         = 0.2,
        horizontal_flip    = True,
        vertical_flip      = True,
        brightness_range   = [0.75, 1.25],
        fill_mode          = 'nearest',
        validation_split   = 0.2,
    )
    if pre_fn is not None:
        aug_kwargs['preprocessing_function'] = pre_fn
        test_datagen = ImageDataGenerator(
            preprocessing_function=pre_fn)
    else:
        test_datagen = ImageDataGenerator()

    train_datagen = ImageDataGenerator(**aug_kwargs)

    common = dict(
        target_size = img_size,
        batch_size  = BATCH_SIZE,
        class_mode  = 'categorical',
        classes     = CLASS_NAMES,
        seed        = SEED,
    )

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR, subset='training',
        shuffle=True,  **common)
    val_gen   = train_datagen.flow_from_directory(
        TRAIN_DIR, subset='validation',
        shuffle=False, **common)
    test_gen  = test_datagen.flow_from_directory(
        TEST_DIR, shuffle=False, **common)

    print(f"   Train:{train_gen.samples} "
          f"Val:{val_gen.samples} "
          f"Test:{test_gen.samples}")
    return train_gen, val_gen, test_gen


# ╔══════════════════════════════════════════╗
# ║  HELPER 3 — BUILD MODEL                 ║
# ╚══════════════════════════════════════════╝

def build_model(model_name):
    cfg         = MODEL_CONFIG[model_name]
    input_shape = (*cfg['img_size'], 3)
    pre_fn      = cfg['preprocess_fn']

    base = cfg['model_fn'](
        weights     = 'imagenet',
        include_top = False,
        input_shape = input_shape,
    )
    base.trainable = False

    inputs = Input(shape=input_shape)
    if pre_fn is None:
        # EfficientNetB0 — keep BN frozen during inference
        x = base(inputs, training=False)
    else:
        x = base(inputs)

    x = GlobalAveragePooling2D(name='gap')(x)

    x = Dense(512, kernel_regularizer=regularizers.l2(1e-4),
               name='fc_512')(x)
    x = BatchNormalization(name='bn1')(x)
    x = Activation('relu', name='relu1')(x)
    x = Dropout(0.5, name='drop1')(x)

    x = Dense(256, kernel_regularizer=regularizers.l2(1e-4),
               name='fc_256')(x)
    x = BatchNormalization(name='bn2')(x)
    x = Activation('relu', name='relu2')(x)
    x = Dropout(0.3, name='drop2')(x)

    x = Dense(128, kernel_regularizer=regularizers.l2(1e-4),
               name='fc_128')(x)
    x = BatchNormalization(name='bn3')(x)
    x = Activation('relu', name='relu3')(x)
    x = Dropout(0.2, name='drop3')(x)

    outputs = Dense(NUM_CLASSES, activation='softmax',
                    name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs,
                  name=f'{model_name}_fixed')
    return model, base


# ╔══════════════════════════════════════════╗
# ║  HELPER 4 — UNFREEZE LAYERS             ║
# ╚══════════════════════════════════════════╝

def unfreeze_layers(base, model_name, phase):
    """
    VGG16/ResNet50 → unfreeze from named layer onwards
    EfficientNetB0 → unfreeze last N layers by count
    BatchNorm → ALWAYS frozen
    """
    cfg           = MODEL_CONFIG[model_name]
    unfreeze_type = cfg['unfreeze_type']
    unfreeze_key  = cfg[f'unfreeze_p{phase}']

    base.trainable = True

    if unfreeze_type == 'named':
        # Unfreeze from named layer onwards
        trainable_flag = False
        for layer in base.layers:
            if layer.name == unfreeze_key:
                trainable_flag = True
            if isinstance(layer,
                          tf.keras.layers.BatchNormalization):
                layer.trainable = False
            else:
                layer.trainable = trainable_flag

        print(f"   Unfreezing from layer '{unfreeze_key}' onwards")

    elif unfreeze_type == 'count':
        # Unfreeze last N layers
        total = len(base.layers)
        for i, layer in enumerate(base.layers):
            if isinstance(layer,
                          tf.keras.layers.BatchNormalization):
                layer.trainable = False
            else:
                layer.trainable = (i >= total - unfreeze_key)

        print(f"   Unfreezing last {unfreeze_key} of "
              f"{total} layers")


# ╔══════════════════════════════════════════╗
# ║  HELPER 5 — CALLBACKS                   ║
# ╚══════════════════════════════════════════╝

def get_callbacks(model_name, phase):
    ckpt = os.path.join(
        OUTPUT_DIR, f'{model_name}_p{phase}_best.keras')
    patience = {1: 10, 2: 12, 3: 10}[phase]
    return [
        ModelCheckpoint(
            ckpt, monitor='val_accuracy',
            save_best_only=True, mode='max', verbose=1),
        EarlyStopping(
            monitor='val_accuracy', patience=patience,
            restore_best_weights=True,
            mode='max', verbose=1),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.3,
            patience=3, min_lr=1e-9, verbose=1),
    ]


# ╔══════════════════════════════════════════╗
# ║  HELPER 6 — TRAINING CURVES             ║
# ╚══════════════════════════════════════════╝

def plot_curves(model_name, *histories):
    acc=[];  val_acc=[]
    loss=[]; val_loss=[]
    splits = []

    for h in histories:
        if h is None: continue
        splits.append(len(acc))
        acc.extend(h.history['accuracy'])
        val_acc.extend(h.history['val_accuracy'])
        loss.extend(h.history['loss'])
        val_loss.extend(h.history['val_loss'])

    epochs = range(1, len(acc)+1)
    clrs   = ['green','blue','orange']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f'{model_name.upper()} — Training History',
                 fontsize=14, fontweight='bold')

    for ax, tr, vl, title in [
        (axes[0], acc,  val_acc,  'Accuracy'),
        (axes[1], loss, val_loss, 'Loss')]:

        ax.plot(epochs, tr, 'b-', label='Train', linewidth=2)
        ax.plot(epochs, vl, 'r-', label='Val',   linewidth=2)
        for i, sp in enumerate(splits[1:], 1):
            ax.axvline(sp+1, color=clrs[i % len(clrs)],
                       linestyle='--', linewidth=1.5,
                       label=f'Phase {i+1}')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(
        OUTPUT_DIR, f'{model_name}_curves.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   💾 Curves: {path}")


# ╔══════════════════════════════════════════╗
# ║  HELPER 7 — EVALUATION                  ║
# ╚══════════════════════════════════════════╝

def evaluate(model, model_name, test_gen):
    test_gen.reset()
    steps  = int(np.ceil(test_gen.samples / BATCH_SIZE))
    probs  = model.predict(test_gen, steps=steps, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_gen.classes[:len(y_pred)]

    overall = accuracy_score(y_true, y_pred)
    target  = MODEL_CONFIG[model_name]['paper_target']
    prev    = PREV_RESULTS[model_name]

    print(f"\n{'─'*55}")
    print(f"  {model_name.upper()} RESULTS")
    print(f"{'─'*55}")
    print(f"  Overall Accuracy : {overall*100:.2f}%")
    print(f"  Paper Target     : {target}%")
    print(f"  Previous         : {prev['overall']}%")
    delta_overall = overall*100 - prev['overall']
    arrow = '⬆️' if delta_overall > 0 else '⬇️'
    print(f"  Change           : {arrow} {abs(delta_overall):.2f}%")

    print(f"\n  Per-Class Accuracy:")
    for i, cls in enumerate(CLASS_NAMES):
        mask = y_true == i
        if mask.sum() == 0: continue
        acc      = accuracy_score(y_true[mask], y_pred[mask])
        prev_cls = prev[cls]
        delta    = acc*100 - prev_cls
        arrow    = '⬆️' if delta > 0 else '⬇️'
        bar      = '█' * int(acc * 20)
        print(f"   {cls:12s}: {acc*100:.2f}%  "
              f"{arrow}{abs(delta):.2f}%  |{bar}|")

    print(f"\n  Classification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES, zero_division=0))

    # Confusion matrix
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES,
                linewidths=0.5, annot_kws={'size': 12})
    ax.set_title(
        f'{model_name.upper()} Confusion Matrix\n'
        f'Accuracy: {overall*100:.2f}%  '
        f'(Target: {target}%)',
        fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = os.path.join(
        OUTPUT_DIR, f'{model_name}_confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   💾 Confusion matrix: {cm_path}")

    return overall, y_true, y_pred


# ╔══════════════════════════════════════════╗
# ║  MAIN TRAINING FUNCTION                 ║
# ╚══════════════════════════════════════════╝

def train_and_evaluate(model_name):
    cfg = MODEL_CONFIG[model_name]
    print(f"\n{'='*55}")
    print(f"  🚀 {model_name.upper()}")
    print(f"{'='*55}")

    # Data
    train_gen, val_gen, test_gen = get_generators(model_name)
    class_weights = compute_class_weights(TRAIN_DIR)

    # Build
    model, base = build_model(model_name)
    trainable   = sum(tf.keras.backend.count_params(w)
                      for w in model.trainable_weights)
    print(f"\n   Total params    : {model.count_params():,}")
    print(f"   Trainable params: {trainable:,}  (head only)")

    # ── PHASE 1: Head only ──────────────────────────────────
    print(f"\n  📌 Phase 1 — head only | "
          f"lr={cfg['phase1_lr']}")
    model.compile(
        optimizer = Adam(learning_rate=cfg['phase1_lr']),
        loss      = tf.keras.losses.CategoricalCrossentropy(
                        label_smoothing=0.1),
        metrics   = ['accuracy'],
    )
    h1 = model.fit(
        train_gen,
        epochs          = cfg['phase1_epochs'],
        validation_data = val_gen,
        class_weight    = class_weights,
        callbacks       = get_callbacks(model_name, 1),
        verbose         = 1,
    )
    p1_best = max(h1.history['val_accuracy'])
    print(f"   ✅ Phase 1 best: {p1_best*100:.2f}%")

    # ── PHASE 2: Fine-tune ──────────────────────────────────
    print(f"\n  📌 Phase 2 — fine-tune | "
          f"lr={cfg['phase2_lr']}")
    unfreeze_layers(base, model_name, phase=2)
    model.compile(
        optimizer = Adam(learning_rate=cfg['phase2_lr']),
        loss      = tf.keras.losses.CategoricalCrossentropy(
                        label_smoothing=0.1),
        metrics   = ['accuracy'],
    )
    h2 = model.fit(
        train_gen,
        epochs          = cfg['phase2_epochs'],
        validation_data = val_gen,
        class_weight    = class_weights,
        callbacks       = get_callbacks(model_name, 2),
        verbose         = 1,
    )
    p2_best = max(h2.history['val_accuracy'])
    print(f"   ✅ Phase 2 best: {p2_best*100:.2f}%")

    # ── PHASE 3: Deeper fine-tune (if needed) ───────────────
    h3      = None
    p3_best = None
    # VGG16 is close → trigger at 96%, others at 95%
    threshold = 0.96 if model_name == 'vgg16' else 0.95

    if p2_best < threshold:
        print(f"\n  📌 Phase 3 — deeper fine-tune | "
              f"lr={cfg['phase3_lr']}")
        unfreeze_layers(base, model_name, phase=3)
        model.compile(
            optimizer = Adam(learning_rate=cfg['phase3_lr']),
            loss      = tf.keras.losses.CategoricalCrossentropy(
                            label_smoothing=0.1),
            metrics   = ['accuracy'],
        )
        h3 = model.fit(
            train_gen,
            epochs          = cfg['phase3_epochs'],
            validation_data = val_gen,
            class_weight    = class_weights,
            callbacks       = get_callbacks(model_name, 3),
            verbose         = 1,
        )
        p3_best = max(h3.history['val_accuracy'])
        print(f"   ✅ Phase 3 best: {p3_best*100:.2f}%")
    else:
        print(f"\n   ⏭️  Phase 3 skipped "
              f"(val={p2_best*100:.2f}% ≥ "
              f"{threshold*100:.0f}%)")

    # Save
    path = os.path.join(
        OUTPUT_DIR, f'{model_name}_final.keras')
    model.save(path)
    print(f"\n   💾 Saved: {path}")

    # Curves
    plot_curves(model_name, h1, h2, h3)

    # Evaluate
    overall, y_true, y_pred = evaluate(
        model, model_name, test_gen)

    # Summary
    all_bests = [p1_best, p2_best]
    if p3_best: all_bests.append(p3_best)

    print(f"\n{'─'*55}")
    print(f"  🏆 {model_name.upper()} SUMMARY")
    print(f"{'─'*55}")
    print(f"  Phase 1 Val     : {p1_best*100:.2f}%")
    print(f"  Phase 2 Val     : {p2_best*100:.2f}%")
    if p3_best:
        print(f"  Phase 3 Val     : {p3_best*100:.2f}%")
    print(f"  Test Accuracy   : {overall*100:.2f}%")
    print(f"  Paper Target    : {cfg['paper_target']}%")
    gap = overall*100 - cfg['paper_target']
    print(f"  Gap vs Paper    : {gap:+.2f}%")
    print(f"{'─'*55}")

    return model, overall


# ╔══════════════════════════════════════════╗
# ║  RUN ALL 3 MODELS                       ║
# ╚══════════════════════════════════════════╝

print("\n" + "="*55)
print("  🧠 TRAINING ALL 3 MODELS — FIXED VERSION")
print("="*55)

all_results = {}

# ── Train VGG16 ──────────────────────────────────────────────
model_vgg16, acc_vgg16 = train_and_evaluate('vgg16')
all_results['vgg16'] = acc_vgg16

# ── Train ResNet50 ───────────────────────────────────────────
model_resnet50, acc_resnet50 = train_and_evaluate('resnet50')
all_results['resnet50'] = acc_resnet50

# ── Train EfficientNetB0 ─────────────────────────────────────
model_effb0, acc_effb0 = train_and_evaluate('efficientnetb0')
all_results['efficientnetb0'] = acc_effb0


# ╔══════════════════════════════════════════╗
# ║  FINAL COMPARISON TABLE                 ║
# ╚══════════════════════════════════════════╝

print("\n" + "="*55)
print("  📊 FINAL COMPARISON — ALL MODELS")
print("="*55)
print(f"  {'Model':<18} {'Previous':>10} "
      f"{'Current':>10} {'Target':>10} {'Gap':>8}")
print(f"  {'─'*56}")

for name, acc in all_results.items():
    prev   = PREV_RESULTS[name]['overall']
    target = MODEL_CONFIG[name]['paper_target']
    delta  = acc*100 - prev
    gap    = acc*100 - target
    arrow  = '⬆️' if delta > 0 else '⬇️'
    status = '✅' if acc*100 >= target else '🔧'
    print(f"  {status} {name:<16} "
          f"{prev:>9.2f}%  "
          f"{acc*100:>9.2f}%  "
          f"{target:>9.2f}%  "
          f"{gap:>+7.2f}%  "
          f"{arrow}{abs(delta):.2f}%")

print("="*55)

# ── List all output files ────────────────────────────────────
print(f"\n  📁 Output files in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath   = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"     📄 {f}  ({size_mb:.1f} MB)")

2026-03-12 10:22:48.260712: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773310968.485668      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773310968.543612      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773310969.067175      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773310969.067215      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773310969.067218      55 computation_placer.cc:177] computation placer alr

✅ Config ready
   GPU: True

  🧠 TRAINING ALL 3 MODELS — FIXED VERSION

  🚀 VGG16
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
   Train:4480 Val:1120 Test:1600

📊 Class weights:
   glioma      : 1.0000 (1400 images)
   meningioma  : 1.0000 (1400 images)
   notumor     : 1.0000 (1400 images)
   pituitary   : 1.0000 (1400 images)


I0000 00:00:1773310996.178157      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773310996.184203      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

   Total params    : 15,145,668
   Trainable params: 429,188  (head only)

  📌 Phase 1 — head only | lr=0.001
Epoch 1/25


I0000 00:00:1773311003.072462     143 service.cc:152] XLA service 0x3de559c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773311003.072504     143 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773311003.072511     143 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773311003.928106     143 cuda_dnn.cc:529] Loaded cuDNN version 91002


...

140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 613ms/step - accuracy: 0.5927 - loss: 1.2111
Epoch 1: val_accuracy improved from -inf to 0.82143, saving model to /kaggle/working/outputs/vgg16_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 128s 790ms/step - accuracy: 0.5934 - loss: 1.2100 - val_accuracy: 0.8214 - val_loss: 0.8469 - learning_rate: 0.0010
Epoch 2/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step - accuracy: 0.7933 - loss: 0.8688
Epoch 2: val_accuracy improved from 0.82143 to 0.85446, saving model to /kaggle/working/outputs/vgg16_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 85s 610ms/step - accuracy: 0.7934 - loss: 0.8686 - val_accuracy: 0.8545 - val_loss: 0.7427 - learning_rate: 0.0010
Epoch 3/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - accuracy: 0.8171 - loss: 0.8270
Epoch 3: val_accuracy improved from 0.85446 to 0.86964, saving model to /kaggle/working/outputs/vgg16_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 84s 597ms/step - accuracy: 0.8172 - loss: 0.8268 - val_accuracy: 0.8696 - val_loss: 

2026-03-12 13:05:17.709567: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 13:05:17.853524: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 13:05:18.206490: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 13:05:18.348009: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 13:05:19.066950: E external/local_xla/xla/stream_

140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.6044 - loss: 1.2149
Epoch 1: val_accuracy improved from -inf to 0.81607, saving model to /kaggle/working/outputs/efficientnetb0_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 115s 615ms/step - accuracy: 0.6050 - loss: 1.2139 - val_accuracy: 0.8161 - val_loss: 0.8267 - learning_rate: 0.0010
Epoch 2/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.7802 - loss: 0.9082
Epoch 2: val_accuracy improved from 0.81607 to 0.85179, saving model to /kaggle/working/outputs/efficientnetb0_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 575ms/step - accuracy: 0.7802 - loss: 0.9082 - val_accuracy: 0.8518 - val_loss: 0.7853 - learning_rate: 0.0010
Epoch 3/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.8237 - loss: 0.8420
Epoch 3: val_accuracy improved from 0.85179 to 0.85804, saving model to /kaggle/working/outputs/efficientnetb0_p1_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 546ms/step - accuracy: 0.8237 - loss: 0.8420 - val_ac

2026-03-12 14:08:11.656761: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 14:08:11.810210: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - accuracy: 0.9210 - loss: 0.6279
Epoch 1: val_accuracy improved from -inf to 0.92143, saving model to /kaggle/working/outputs/efficientnetb0_p3_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 118s 612ms/step - accuracy: 0.9210 - loss: 0.6279 - val_accuracy: 0.9214 - val_loss: 0.6041 - learning_rate: 5.0000e-06
Epoch 2/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step - accuracy: 0.9345 - loss: 0.6091
Epoch 2: val_accuracy improved from 0.92143 to 0.92768, saving model to /kaggle/working/outputs/efficientnetb0_p3_best.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 584ms/step - accuracy: 0.9345 - loss: 0.6091 - val_accuracy: 0.9277 - val_loss: 0.5919 - learning_rate: 5.0000e-06
Epoch 3/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step - accuracy: 0.9259 - loss: 0.6158
Epoch 3: val_accuracy did not improve from 0.92768
140/140 ━━━━━━━━━━━━━━━━━━━━ 79s 564ms/step - accuracy: 0.9259 - loss: 0.6158 - val_accuracy: 0.9277 - val_loss: 0.6018 - learning_rate: 5.0000e-06
Epoc

In [2]:

# ╔══════════════════════════════════════════════════╗
# ║  SECTION 2 — LIME                               ║
# ╠══════════════════════════════════════════════════╣
# ║  Steps (paper equations):                       ║
# ║  Step1: Quickshift → superpixels {S1..Sn}       ║
# ║  Eq15:  Perturb + π(x,z)=exp(-d²/σ²)           ║
# ║  Eq16:  Local model f(z')= wᵀz' + b            ║
# ║  Eq17:  min Σ π·[f(x)−f(z')]²                  ║
# ║  Step4: Rank → green=positive red=negative      ║
# ╚══════════════════════════════════════════════════╝

def make_lime_predict_fn(model, model_name):
    """
    LIME passes uint8 images (0-255).
    Apply correct preprocessing before model.predict().
    """
    fn = get_preprocess_fn(model_name)

    def predict_fn(images_uint8):
        imgs = np.array(images_uint8, dtype=np.float32)
        if fn is not None:
            # VGG16/ResNet50: apply their preprocessor
            imgs = np.stack(
                [fn(img) for img in imgs], axis=0)
        # EfficientNetB0: pass as-is (0-255)
        return model.predict(imgs, verbose=0)

    return predict_fn


def run_lime_model(model, model_name,
                   test_dir=TEST_DIR,
                   output_dir=OUTPUT_DIR,
                   samples_per_class=2,
                   num_samples=1000,
                   num_features=5):
    """
    Full LIME pipeline for all 4 classes.
    Grid layout (5 cols per sample):
      Col0: Original + prediction
      Col1: Quickshift superpixels    ← Step1
      Col2: Top positive (green)      ← Step4
      Col3: Pos(green) + Neg(red)     ← Step4
      Col4: Weight heatmap            ← Eq16 weights
    """
    print(f"\n{'─'*55}")
    print(f"  🍋 LIME: {model_name.upper()}")
    print(f"   Samples/class={samples_per_class} "
          f"| num_samples={num_samples} "
          f"| features={num_features}")

    predict_fn = make_lime_predict_fn(model, model_name)

    # Quickshift segmenter — paper requirement
    segmenter  = SegmentationAlgorithm(
        'quickshift',
        kernel_size = 4,
        max_dist    = 200,
        ratio       = 0.2,
    )
    explainer  = lime_lib.LimeImageExplainer(
                     random_state=SEED)
    fn         = get_preprocess_fn(model_name)

    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 5
    fig    = plt.figure(
                figsize=(4 * n_cols, 5 * n_rows))
    gs     = gridspec.GridSpec(
                n_rows, n_cols,
                wspace=0.1, hspace=0.4)

    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = get_image_files(
                        cls_dir, samples_per_class)

        if not img_files:
            print(f"   ⚠️  No images: {cls_dir}")
            continue

        for s, img_path in enumerate(img_files):
            col = s * 5
            try:
                # Load uint8 for LIME + display
                img_uint8 = load_as_uint8(
                                img_path, model_name)

                # Prediction for title
                pred_idx, pred_conf, pred_label = \
                    get_prediction(
                        model, img_path, model_name)
                correct   = pred_label == cls_name
                t_color   = 'green' if correct else 'red'

                # ── Step1+Eq15+Eq16+Eq17: LIME ──────────
                explanation = explainer.explain_instance(
                    img_uint8,
                    predict_fn,
                    top_labels      = NUM_CLASSES,
                    hide_color      = 0,
                    num_samples     = num_samples,
                    segmentation_fn = segmenter,
                    random_seed     = SEED,
                )

                # ── Col 0: Original ──────────────────────
                ax0 = fig.add_subplot(gs[row, col])
                ax0.imshow(img_uint8)
                ax0.set_title(
                    f"True: {cls_name}\n"
                    f"Pred: {pred_label} "
                    f"({pred_conf:.1f}%)",
                    fontsize=8, color=t_color)
                ax0.axis('off')

                # ── Col 1: Quickshift superpixels ────────
                ax1 = fig.add_subplot(gs[row, col+1])
                segs = quickshift(
                    img_uint8,
                    kernel_size=4,
                    max_dist=200,
                    ratio=0.2)
                n_segs = len(np.unique(segs))
                ax1.imshow(mark_boundaries(
                    img_uint8.astype(np.float32)/255.0,
                    segs,
                    color=(1,1,0), mode='thick'))
                ax1.set_title(
                    f'Quickshift\n({n_segs} segments)',
                    fontsize=8)
                ax1.axis('off')

                # ── Step4: Get masks ─────────────────────

                # ── Col 2: Top positive (green) ──────────
                try:
                    temp_pos, mask_pos = \
                        explanation.get_image_and_mask(
                            label         = pred_idx,
                            positive_only = True,
                            num_features  = num_features,
                            hide_rest     = False)
                    ax2 = fig.add_subplot(gs[row, col+2])
                    ax2.imshow(mark_boundaries(
                        temp_pos.astype(
                            np.float32)/255.0,
                        mask_pos,
                        color=(0,1,0), mode='thick'))
                    ax2.set_title(
                        f'Positive\n(top {num_features})',
                        fontsize=8)
                    ax2.axis('off')
                except Exception as e2:
                    ax2 = fig.add_subplot(gs[row, col+2])
                    ax2.text(0.5, 0.5,
                             f'N/A\n{str(e2)[:25]}',
                             ha='center', va='center',
                             fontsize=7,
                             transform=ax2.transAxes)
                    ax2.axis('off')

                # ── Col 3: Pos(green) + Neg(red) ─────────
                try:
                    temp_pn, mask_pn = \
                        explanation.get_image_and_mask(
                            label         = pred_idx,
                            positive_only = False,
                            num_features  = num_features,
                            hide_rest     = False)
                    _, mask_pos_only = \
                        explanation.get_image_and_mask(
                            label         = pred_idx,
                            positive_only = True,
                            num_features  = num_features,
                            hide_rest     = False)

                    neg_mask = (
                        (mask_pn == 1) &
                        (mask_pos_only == 0))

                    disp = temp_pn.astype(
                               np.float32) / 255.0
                    disp = mark_boundaries(
                               disp, mask_pos_only,
                               color=(0,1,0),
                               mode='thick')
                    disp = mark_boundaries(
                               disp,
                               neg_mask.astype(int),
                               color=(1,0,0),
                               mode='thick')
                    ax3 = fig.add_subplot(gs[row, col+3])
                    ax3.imshow(disp)
                    ax3.set_title(
                        'Pos(green)\n+Neg(red)',
                        fontsize=8)
                    ax3.axis('off')
                except Exception as e3:
                    ax3 = fig.add_subplot(gs[row, col+3])
                    ax3.text(0.5, 0.5,
                             f'N/A\n{str(e3)[:25]}',
                             ha='center', va='center',
                             fontsize=7,
                             transform=ax3.transAxes)
                    ax3.axis('off')

                # ── Col 4: Weight heatmap ─────────────────
                try:
                    w_dict     = dict(
                        explanation.local_exp[pred_idx])
                    weight_map = np.vectorize(
                        lambda sid: w_dict.get(sid, 0.0)
                    )(explanation.segments)

                    vmax = float(
                        np.max(np.abs(weight_map)))
                    vmax = vmax if vmax > 1e-8 else 1.0

                    ax4 = fig.add_subplot(gs[row, col+4])
                    im  = ax4.imshow(
                        weight_map,
                        cmap='RdYlGn',
                        vmin=-vmax, vmax=vmax)
                    ax4.set_title(
                        'Weight Map\n(learned w)',
                        fontsize=8)
                    ax4.axis('off')
                    plt.colorbar(
                        im, ax=ax4,
                        fraction=0.046, pad=0.04)
                except Exception as e4:
                    ax4 = fig.add_subplot(gs[row, col+4])
                    ax4.text(0.5, 0.5,
                             f'N/A\n{str(e4)[:25]}',
                             ha='center', va='center',
                             fontsize=7,
                             transform=ax4.transAxes)
                    ax4.axis('off')

            except Exception as e:
                print(f"   ⚠️  Failed "
                      f"{cls_name}[{s}]: {e}")
                for offset in range(5):
                    ax = fig.add_subplot(
                             gs[row, col+offset])
                    ax.text(0.5, 0.5,
                            f'Error:\n{str(e)[:35]}',
                            ha='center', va='center',
                            fontsize=7, color='red',
                            transform=ax.transAxes)
                    ax.axis('off')

    fig.suptitle(
        f'{model_name.upper()} — LIME Explanations\n'
        f'Quickshift | Green=supports | Red=contradicts',
        fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()

    save_path = os.path.join(
        output_dir, f'{model_name}_lime.png')
    plt.savefig(save_path, dpi=150,
                bbox_inches='tight')
    plt.close(fig)
    print(f"   💾 Saved: {save_path}")


def run_lime_all(models_dict=LOADED_MODELS,
                 test_dir=TEST_DIR,
                 output_dir=OUTPUT_DIR,
                 samples_per_class=2,
                 num_samples=1000,
                 num_features=5):
    """Run LIME for all loaded models."""
    print("\n" + "="*55)
    print("  🍋 RUNNING LIME — ALL MODELS")
    print("="*55)
    for name, model in models_dict.items():
        run_lime_model(model, name,
                       test_dir, output_dir,
                       samples_per_class,
                       num_samples, num_features)
    print("\n✅ LIME complete.")


# ╔══════════════════════════════════════════════════╗
# ║  SECTION 3 — SUPERPIXEL WEIGHT CHART            ║
# ║  Paper Figure 4                                 ║
# ╚══════════════════════════════════════════════════╝

def plot_superpixel_weights(model, model_name,
                             img_path,
                             output_dir=OUTPUT_DIR,
                             num_samples=1000,
                             top_n=15):
    """
    Bar chart of top-N superpixel weights.
    Green = supports prediction
    Red   = contradicts prediction
    """
    predict_fn = make_lime_predict_fn(model, model_name)
    segmenter  = SegmentationAlgorithm(
        'quickshift',
        kernel_size=4, max_dist=200, ratio=0.2)
    explainer  = lime_lib.LimeImageExplainer(
                     random_state=SEED)

    img_uint8  = load_as_uint8(img_path, model_name)
    pred_idx, pred_conf, pred_label = \
        get_prediction(model, img_path, model_name)

    explanation = explainer.explain_instance(
        img_uint8, predict_fn,
        top_labels=NUM_CLASSES, hide_color=0,
        num_samples=num_samples,
        segmentation_fn=segmenter, random_seed=SEED)

    weights = sorted(
        explanation.local_exp[pred_idx],
        key=lambda x: abs(x[1]), reverse=True)[:top_n]

    seg_ids = [f'S{w[0]}' for w in weights]
    values  = [w[1]        for w in weights]
    colours = ['#2ca02c' if v > 0 else '#d62728'
               for v in values]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'{model_name.upper()} — Superpixel Weights\n'
        f'Predicted: {pred_label} ({pred_conf:.1f}%)',
        fontsize=13, fontweight='bold')

    # Bar chart
    axes[0].barh(seg_ids[::-1], values[::-1],
                 color=colours[::-1],
                 edgecolor='white', height=0.6)
    axes[0].axvline(0, color='black', linewidth=0.8)
    axes[0].set_xlabel('Weight (w)', fontsize=11)
    axes[0].set_ylabel('Superpixel ID', fontsize=11)
    axes[0].set_title(f'Top {top_n} Weights',
                      fontsize=11)
    axes[0].grid(axis='x', alpha=0.3)

    # Weight map on image
    segs = quickshift(img_uint8, kernel_size=4,
                      max_dist=200, ratio=0.2)
    w_dict     = dict(explanation.local_exp[pred_idx])
    weight_map = np.vectorize(
        lambda sid: w_dict.get(sid, 0.0))(segs)
    vmax = float(np.max(np.abs(weight_map))) or 1.0

    im = axes[1].imshow(weight_map, cmap='RdYlGn',
                        vmin=-vmax, vmax=vmax)
    axes[1].set_title('Weight Map on Image', fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1],
                 fraction=0.046, pad=0.04)

    plt.tight_layout()
    fname = (f'{model_name}_superpixel_weights_'
             f'{pred_label}.png')
    path  = os.path.join(output_dir, fname)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   💾 Superpixel chart: {path}")


def run_superpixel_weights_all(
        models_dict=LOADED_MODELS,
        test_dir=TEST_DIR,
        output_dir=OUTPUT_DIR,
        num_samples=1000,
        top_n=15):
    """Run superpixel weight chart for all models."""
    print("\n" + "="*55)
    print("  📊 SUPERPIXEL WEIGHT CHARTS")
    print("="*55)
    for model_name, model in models_dict.items():
        # One sample per class
        for cls_name in CLASS_NAMES:
            cls_dir   = os.path.join(test_dir, cls_name)
            img_files = get_image_files(cls_dir, n=1)
            if not img_files:
                continue
            try:
                print(f"   {model_name} / {cls_name}")
                plot_superpixel_weights(
                    model, model_name,
                    img_files[0], output_dir,
                    num_samples, top_n)
            except Exception as e:
                print(f"   ⚠️  Failed: {e}")


# ╔══════════════════════════════════════════════════╗
# ║  RUN EVERYTHING                                 ║
# ╚══════════════════════════════════════════════════╝

# ── Phase 1: Grad-CAM ────────────────────────────────────────
run_gradcam_all(
    models_dict       = LOADED_MODELS,
    test_dir          = TEST_DIR,
    output_dir        = OUTPUT_DIR,
    samples_per_class = 2,
)

# ── Phase 2: LIME ────────────────────────────────────────────
run_lime_all(
    models_dict       = LOADED_MODELS,
    test_dir          = TEST_DIR,
    output_dir        = OUTPUT_DIR,
    samples_per_class = 2,
    num_samples       = 1000,
    num_features      = 5,
)

# ── Phase 3: Superpixel weight charts ────────────────────────
run_superpixel_weights_all(
    models_dict = LOADED_MODELS,
    test_dir    = TEST_DIR,
    output_dir  = OUTPUT_DIR,
    num_samples = 1000,
    top_n       = 15,
)

# ── List all saved files ─────────────────────────────────────
print("\n" + "="*55)
print("  🎉 XAI COMPLETE — All files saved:")
print("="*55)
xai_files = [
    f for f in sorted(os.listdir(OUTPUT_DIR))
    if any(x in f for x in [
        'gradcam', 'lime', 'superpixel'])
]
for f in xai_files:
    path    = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"   📄 {f}  ({size_mb:.2f} MB)")
print("="*55)

✅ XAI setup ready
   GPU: True

📂 Checking model files:
   ✅ vgg16: /kaggle/working/outputs/vgg16_final.keras
   ✅ resnet50: /kaggle/working/outputs/resnet50_final.keras
   ✅ efficientnetb0: /kaggle/working/outputs/efficientnetb0_final.keras

📂 Loading vgg16...
   ✅ Loaded: vgg16

📂 Loading resnet50...
   ✅ Loaded: resnet50

📂 Loading efficientnetb0...
   ✅ Loaded: efficientnetb0

  🔥 RUNNING GRAD-CAM — ALL MODELS

───────────────────────────────────────────────────────
  🔥 Grad-CAM: VGG16
   Last conv layer: block5_conv3
   ⚠️  Failed glioma[0]: Input 0 of layer "functional" is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(None, 7, 7, 512)
   ⚠️  Failed glioma[1]: Input 0 of layer "functional_1" is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(None, 7, 7, 512)
   ⚠️  Failed meningioma[0]: Input 0 of layer "functional_2" is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(None, 7, 7, 512)
   ⚠️

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

...

   💾 Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_meningioma.png
   efficientnetb0 / notumor


  0%|          | 0/1000 [00:00<?, ?it/s]

   💾 Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_notumor.png
   efficientnetb0 / pituitary


  0%|          | 0/1000 [00:00<?, ?it/s]

   💾 Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_pituitary.png

  🎉 XAI COMPLETE — All files saved:
   📄 efficientnetb0_gradcam.png  (0.08 MB)
   📄 efficientnetb0_lime.png  (2.84 MB)
   📄 efficientnetb0_superpixel_weights_glioma.png  (0.11 MB)
   📄 efficientnetb0_superpixel_weights_meningioma.png  (0.10 MB)
   📄 efficientnetb0_superpixel_weights_notumor.png  (0.11 MB)
   📄 efficientnetb0_superpixel_weights_pituitary.png  (0.11 MB)
   📄 resnet50_gradcam.png  (0.08 MB)
   📄 resnet50_lime.png  (2.84 MB)
   📄 resnet50_superpixel_weights_meningioma.png  (0.09 MB)
   📄 resnet50_superpixel_weights_notumor.png  (0.11 MB)
   📄 resnet50_superpixel_weights_pituitary.png  (0.12 MB)
   📄 vgg16_gradcam.png  (0.08 MB)
   📄 vgg16_lime.png  (2.77 MB)
   📄 vgg16_superpixel_weights_glioma.png  (0.10 MB)
   📄 vgg16_superpixel_weights_meningioma.png  (0.09 MB)
   📄 vgg16_superpixel_weights_notumor.png  (0.12 MB)
   📄 vgg16_superpixel_weights_pituitary.png  (0.11 MB)


In [12]:
# ── Run this ONCE — prints exact tensor info ─────────────────
import tensorflow as tf

model = tf.keras.models.load_model(
    '/kaggle/working/outputs/vgg16_final.keras')

print("=" * 60)
print("model.inputs (list):")
for i, t in enumerate(model.inputs):
    print(f"  [{i}] type={type(t)}  shape={t.shape}  name={t.name}")

print("\nmodel.outputs (list):")
for i, t in enumerate(model.outputs):
    print(f"  [{i}] type={type(t)}  shape={t.shape}  name={t.name}")

print("\nmodel.input (raw):", type(model.input))
print("model.output (raw):", type(model.output))

# Base layer
base = None
for layer in model.layers:
    if hasattr(layer, 'layers') and len(layer.layers) > 5:
        base = layer
        break

print(f"\nBase layer: {base.name}")
print("base.inputs (list):")
for i, t in enumerate(base.inputs):
    print(f"  [{i}] type={type(t)}  shape={t.shape}  name={t.name}")

print("\nbase.input (raw):", type(base.input))

# Find last conv
for layer in reversed(base.layers):
    if isinstance(layer, (tf.keras.layers.Conv2D,
                           tf.keras.layers.DepthwiseConv2D)):
        conv = base.get_layer(layer.name)
        print(f"\nLast conv: {layer.name}")
        print(f"conv.output type : {type(conv.output)}")
        print(f"conv.output shape: {conv.output.shape}")
        break

model.inputs (list):
  [0] type=<class 'keras.src.backend.common.keras_tensor.KerasTensor'>  shape=(None, 224, 224, 3)  name=input_layer_1

model.outputs (list):
  [0] type=<class 'keras.src.backend.common.keras_tensor.KerasTensor'>  shape=(None, 4)  name=keras_tensor_8640

model.input (raw): <class 'list'>
model.output (raw): <class 'keras.src.backend.common.keras_tensor.KerasTensor'>

Base layer: vgg16
base.inputs (list):
  [0] type=<class 'keras.src.backend.common.keras_tensor.KerasTensor'>  shape=(None, 224, 224, 3)  name=input_layer

base.input (raw): <class 'list'>

Last conv: block5_conv3
conv.output type : <class 'keras.src.backend.common.keras_tensor.KerasTensor'>
conv.output shape: (None, 14, 14, 512)


In [16]:
# =============================================================
# 🔧 RESNET50 GRAD-CAM FIX
# Problem: conv5_block3_3_conv output goes through Add layer
#          → gradient path broken → grads None
# Fix: use conv5_block3_out (after Add+BN+ReLU) for ResNet50
#      This is the true last feature tensor before GAP
# =============================================================

import os, glob, cv2, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.applications.vgg16    import preprocess_input as vgg16_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_pre
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
OUTPUT_DIR  = '/kaggle/working/outputs'
TEST_DIR    = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_SETTINGS = {
    'vgg16': {
        'img_size'      : (224, 224),
        'preprocess_fn' : vgg16_pre,
        'model_path'    : f'{OUTPUT_DIR}/vgg16_final.keras',
        # Last Conv2D — works fine
        'conv_layer'    : None,   # auto-detect
    },
    'resnet50': {
        'img_size'      : (224, 224),
        'preprocess_fn' : resnet50_pre,
        'model_path'    : f'{OUTPUT_DIR}/resnet50_final.keras',
        # ✅ Use conv5_block3_out — AFTER the Add layer
        # This is the true output of the last residual block
        'conv_layer'    : 'conv5_block3_out',
    },
    'efficientnetb0': {
        'img_size'      : (224, 224),
        'preprocess_fn' : None,
        'model_path'    : f'{OUTPUT_DIR}/efficientnetb0_final.keras',
        # Last Conv2D — works fine
        'conv_layer'    : None,   # auto-detect
    },
}

print("✅ Config ready")
print(f"   GPU: {len(tf.config.list_physical_devices('GPU'))>0}")


# ╔══════════════════════════════════════════════════════╗
# ║  IMAGE HELPERS                                      ║
# ╚══════════════════════════════════════════════════════╝

def get_image_files(cls_dir, n=2):
    return sorted(
        glob.glob(f'{cls_dir}/*.jpg')  +
        glob.glob(f'{cls_dir}/*.jpeg') +
        glob.glob(f'{cls_dir}/*.png')
    )[:n]

def load_for_predict(img_path, model_name):
    sz  = MODEL_SETTINGS[model_name]['img_size']
    fn  = MODEL_SETTINGS[model_name]['preprocess_fn']
    img = keras_image.load_img(img_path, target_size=sz)
    arr = keras_image.img_to_array(img)
    if fn is not None:
        arr = fn(arr)
    return np.expand_dims(arr, 0).astype(np.float32)

def load_rgb_display(img_path, model_name):
    sz  = MODEL_SETTINGS[model_name]['img_size']
    bgr = cv2.imread(img_path)
    return cv2.cvtColor(
        cv2.resize(bgr, sz), cv2.COLOR_BGR2RGB)


# ╔══════════════════════════════════════════════════════╗
# ║  GET BASE LAYER + TARGET LAYER                      ║
# ║                                                     ║
# ║  VGG16       → last Conv2D (no skip connections)   ║
# ║  ResNet50    → conv5_block3_out (after Add layer)  ║
# ║  EffNetB0    → last Conv2D (top_conv)              ║
# ╚══════════════════════════════════════════════════════╝

def get_base_and_target_layer(model, model_name):
    # Find base sub-model
    base_layer = None
    for layer in model.layers:
        if hasattr(layer, 'layers') and \
           len(layer.layers) > 5:
            base_layer = layer
            break
    if base_layer is None:
        raise ValueError("Base model not found.")

    # Check for manually specified layer
    forced = MODEL_SETTINGS[model_name]['conv_layer']
    if forced is not None:
        # Verify it exists
        layer_names = [l.name for l in base_layer.layers]
        if forced in layer_names:
            print(f"   Base     : {base_layer.name}")
            print(f"   Target   : {forced} (forced) ✅")
            return base_layer, forced
        else:
            print(f"   ⚠️  Forced layer '{forced}' not found")
            print(f"   Available (last 10):")
            for n in layer_names[-10:]:
                print(f"      {n}")

    # Auto-detect: last Conv2D
    conv_name = None
    for layer in reversed(base_layer.layers):
        if isinstance(layer, (
                tf.keras.layers.Conv2D,
                tf.keras.layers.DepthwiseConv2D)):
            conv_name = layer.name
            break

    if conv_name is None:
        raise ValueError("No Conv2D found.")

    print(f"   Base     : {base_layer.name} "
          f"({len(base_layer.layers)} layers)")
    print(f"   Target   : {conv_name} (auto-detect)")
    return base_layer, conv_name


# ╔══════════════════════════════════════════════════════╗
# ║  COMPUTE GRAD-CAM — FRESH GRAPH EVERY CALL         ║
# ╚══════════════════════════════════════════════════════╝

def compute_gradcam_direct(model, base_layer,
                            target_layer_name,
                            img_array,
                            pred_index=None):
    """
    Fresh conv_fn built every call → no cached tensors
    tf.Variable input → always watched by tape
    persistent tape → reliable gradient
    """
    # Build fresh conv extractor every call
    base_in    = base_layer.inputs[0]
    target_out = base_layer.get_layer(
                     target_layer_name).output
    conv_fn    = tf.keras.models.Model(
        inputs  = base_in,
        outputs = target_out
    )

    # tf.Variable → always watched
    img_var = tf.Variable(
        img_array, trainable=True, dtype=tf.float32)

    with tf.GradientTape(persistent=True) as tape:
        tape.watch(img_var)

        conv_features = conv_fn(
            img_var, training=False)
        tape.watch(conv_features)

        predictions = model(
            img_var, training=False)

        if pred_index is None:
            pred_index = int(
                tf.argmax(predictions[0]).numpy())

        class_score = predictions[:, pred_index]

    # Try conv gradient first
    grads = tape.gradient(class_score, conv_features)

    # Fallback: input gradient
    if grads is None:
        grads = tape.gradient(class_score, img_var)
        if grads is not None:
            grads = tf.reduce_mean(
                tf.abs(grads), axis=-1, keepdims=True)
            grads = tf.image.resize(
                grads,
                [conv_features.shape[1],
                 conv_features.shape[2]])
            grads = tf.repeat(
                grads,
                conv_features.shape[-1], axis=-1)

    del tape

    if grads is None:
        weights = tf.reduce_mean(
            conv_features, axis=(0, 1, 2))
    else:
        weights = tf.reduce_mean(grads, axis=(0, 1, 2))

    cam = tf.reduce_sum(
        tf.multiply(weights, conv_features[0]), axis=-1)
    cam = tf.nn.relu(cam)

    cam_min = tf.reduce_min(cam)
    cam_max = tf.reduce_max(cam)
    if (cam_max - cam_min) > 1e-8:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        cam = tf.zeros_like(cam)

    pred_conf = float(
        predictions[0][pred_index].numpy()) * 100

    return cam.numpy(), int(pred_index), pred_conf


# ╔══════════════════════════════════════════════════════╗
# ║  VERIFY RESNET50 LAYER NAME                         ║
# ╚═══════════════════════════════════��══════════════════╝

def verify_resnet_layer(model):
    """Print last 15 layer names of ResNet50 sub-model."""
    for layer in model.layers:
        if hasattr(layer, 'layers') and \
           'resnet' in layer.name.lower():
            print(f"\n   Last 15 layers of {layer.name}:")
            for l in layer.layers[-15:]:
                try:
                    shape = l.output_shape
                except Exception:
                    shape = "multiple"
                print(f"      {l.name:40s} {str(shape)}")
            return layer
    return None


# ╔══════════════════════════════════════════════════════╗
# ║  OVERLAY                                            ║
# ╚══════════════════════════════════════════════════════╝

def build_overlay(orig_rgb, heatmap, alpha=0.5):
    H, W     = orig_rgb.shape[:2]
    hm_up    = cv2.resize(
        heatmap.astype(np.float32), (W, H),
        interpolation=cv2.INTER_LINEAR)
    hm_up    = np.clip(hm_up, 0, 1)
    hm_uint8 = np.uint8(255 * hm_up)
    hm_jet   = cv2.applyColorMap(
        hm_uint8, cv2.COLORMAP_JET)
    hm_jet   = cv2.cvtColor(hm_jet, cv2.COLOR_BGR2RGB)
    overlay  = np.uint8(
        hm_jet * alpha + orig_rgb * (1 - alpha))
    return overlay, hm_up


# ╔══════════════════════════════════════════════════════╗
# ║  RUN FOR ONE MODEL                                  ║
# ╚══════════════════════════════════════════════════════╝

def run_gradcam_model(model, model_name,
                       test_dir=TEST_DIR,
                       output_dir=OUTPUT_DIR,
                       samples_per_class=2):
    print(f"\n{'─'*55}")
    print(f"  🔥 Grad-CAM: {model_name.upper()}")

    try:
        base_layer, target_layer = \
            get_base_and_target_layer(model, model_name)
    except Exception as e:
        print(f"   ❌ Setup failed: {e}")
        return

    # Sanity check
    first_dir  = os.path.join(test_dir, CLASS_NAMES[0])
    test_files = get_image_files(first_dir, 1)
    if test_files:
        arr        = load_for_predict(
            test_files[0], model_name)
        hm, pi, pc = compute_gradcam_direct(
            model, base_layer, target_layer, arr)
        ok = (hm.max() > hm.min())
        print(f"   {'✅ GRADS OK' if ok else '⚠️  FALLBACK'}"
              f" | {CLASS_NAMES[pi]} ({pc:.1f}%)"
              f" | [{hm.min():.3f},{hm.max():.3f}]")

    # Figure
    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 3
    fig    = plt.figure(
        figsize=(5 * n_cols, 5 * n_rows))
    gs     = gridspec.GridSpec(
        n_rows, n_cols,
        wspace=0.05, hspace=0.4)

    ok_count   = 0
    fail_count = 0

    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = get_image_files(
            cls_dir, samples_per_class)
        if not img_files:
            continue

        for s, img_path in enumerate(img_files):
            col = s * 3
            try:
                img_arr  = load_for_predict(
                    img_path, model_name)
                orig_rgb = load_rgb_display(
                    img_path, model_name)

                heatmap, pred_idx, pred_conf = \
                    compute_gradcam_direct(
                        model, base_layer,
                        target_layer, img_arr)

                if heatmap.max() > heatmap.min():
                    ok_count   += 1
                else:
                    fail_count += 1

                overlay, hm_up = build_overlay(
                    orig_rgb, heatmap)

                pred_label = CLASS_NAMES[pred_idx]
                correct    = (pred_label == cls_name)
                t_color    = 'green' if correct \
                             else 'red'

                ax0 = fig.add_subplot(gs[row, col])
                ax0.imshow(orig_rgb)
                ax0.set_title(
                    f"True: {cls_name}\n"
                    f"Pred: {pred_label} "
                    f"({pred_conf:.1f}%)",
                    fontsize=9, color=t_color,
                    fontweight='bold')
                ax0.axis('off')

                ax1 = fig.add_subplot(gs[row, col+1])
                im1 = ax1.imshow(
                    hm_up, cmap='jet',
                    vmin=0, vmax=1)
                ax1.set_title(
                    'Grad-CAM Heatmap', fontsize=9)
                ax1.axis('off')
                plt.colorbar(im1, ax=ax1,
                             fraction=0.046, pad=0.04)

                ax2 = fig.add_subplot(gs[row, col+2])
                ax2.imshow(overlay)
                ax2.set_title(
                    'Heatmap on MRI', fontsize=9)
                ax2.axis('off')

            except Exception as e:
                fail_count += 1
                print(f"   ⚠️  {cls_name}[{s}]: {e}")
                for off in range(3):
                    ax = fig.add_subplot(
                        gs[row, col+off])
                    ax.text(0.5, 0.5,
                            f'Error:\n{str(e)[:40]}',
                            ha='center', va='center',
                            fontsize=7, color='red',
                            transform=ax.transAxes)
                    ax.axis('off')

    total = ok_count + fail_count
    print(f"   📊 {ok_count}/{total} GRADS OK "
          f"| {fail_count}/{total} fallback")

    fig.suptitle(
        f'{model_name.upper()} — Grad-CAM\n'
        f'Green=Correct | Red=Wrong',
        fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(
        output_dir, f'{model_name}_gradcam.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   💾 Saved: {path}")


# ╔══════════════════════════════════════════════════════╗
# ║  LOAD + RUN ALL                                     ║
# ╚══════════════════════════════════════════════════════╝

print("\n📂 Checking model files:")
available = {}
for name, cfg in MODEL_SETTINGS.items():
    path   = cfg['model_path']
    exists = os.path.exists(path)
    print(f"   {'✅' if exists else '❌'} "
          f"{name}: {path}")
    if exists:
        available[name] = path

if not available:
    print("❌ No model files found!")
else:
    print("\n📂 Loading models...")
    loaded = {}
    for name, path in available.items():
        print(f"   Loading {name}...")
        loaded[name] = tf.keras.models.load_model(path)
        print(f"   ✅ Loaded: {name}")

    # Verify ResNet50 layer names before running
    print("\n🔍 Verifying ResNet50 layer names...")
    if 'resnet50' in loaded:
        verify_resnet_layer(loaded['resnet50'])

    print("\n" + "="*55)
    print("  🔥 GRAD-CAM — ALL MODELS")
    print("="*55)

    for model_name, model in loaded.items():
        run_gradcam_model(
            model             = model,
            model_name        = model_name,
            test_dir          = TEST_DIR,
            output_dir        = OUTPUT_DIR,
            samples_per_class = 2,
        )

    print("\n" + "="*55)
    print("  🎉 GRAD-CAM COMPLETE")
    print("="*55)
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if 'gradcam' in f:
            p    = os.path.join(OUTPUT_DIR, f)
            size = os.path.getsize(p) / (1024*1024)
            print(f"   📄 {f}  ({size:.2f} MB)")
    print("="*55)

✅ Config ready
   GPU: True

📂 Checking model files:
   ✅ vgg16: /kaggle/working/outputs/vgg16_final.keras
   ✅ resnet50: /kaggle/working/outputs/resnet50_final.keras
   ✅ efficientnetb0: /kaggle/working/outputs/efficientnetb0_final.keras

📂 Loading models...
   Loading vgg16...
   ✅ Loaded: vgg16
   Loading resnet50...
   ✅ Loaded: resnet50
   Loading efficientnetb0...
   ✅ Loaded: efficientnetb0

🔍 Verifying ResNet50 layer names...

   Last 15 layers of resnet50:
      conv5_block2_2_relu                      multiple
      conv5_block2_3_conv                      multiple
      conv5_block2_3_bn                        multiple
      conv5_block2_add                         multiple
      conv5_block2_out                         multiple
      conv5_block3_1_conv                      multiple
      conv5_block3_1_bn                        multiple
      conv5_block3_1_relu                      multiple
      conv5_block3_2_conv                      multiple
      conv5_block3_2_bn   